In [2]:
# ============================================================
# CELL 1: Imports + robust geometry loading
# ============================================================

from __future__ import annotations

import os
import math
import time
import json
import random
import warnings
from collections import defaultdict
from typing import Dict, List, Optional, Tuple

import numpy as np

from shapely.geometry import (
    Polygon,
    MultiPolygon,
    LineString,
    MultiLineString,
)
from shapely import wkt as shapely_wkt

warnings.filterwarnings("ignore")


# ------------------------------------------------------------
# Helpers for geometry conversion
# ------------------------------------------------------------

def _linestring_to_polygon(geom):
    """
    Convert a LineString that represents a ring into a Polygon.
    Safely handles empty / degenerate line strings.
    """
    if geom is None or geom.is_empty:
        raise ValueError("empty LineString")

    coords = list(geom.coords)

    if len(coords) == 0:
        raise ValueError("LineString has no coordinates")

    # Need at least 3 points before closure
    if len(coords) < 3:
        raise ValueError(f"LineString too short for polygon conversion: {len(coords)} coords")

    if coords[0] != coords[-1]:
        coords.append(coords[0])   # force close the ring

    # Polygon ring needs at least 4 coordinates after closing
    if len(coords) < 4:
        raise ValueError(f"closed ring too short: {len(coords)} coords")

    poly = Polygon(coords)

    if poly.is_empty:
        raise ValueError("converted polygon is empty")

    if not poly.is_valid:
        poly = poly.buffer(0)

    if poly.is_empty:
        raise ValueError("polygon became empty after validity fix")

    return poly


def _geom_to_polygon(geom):
    """
    Convert supported geometry types into Polygon / MultiPolygon.
    """
    if geom is None or geom.is_empty:
        raise ValueError("empty geometry")

    if geom.geom_type in ("Polygon", "MultiPolygon"):
        return geom

    if geom.geom_type == "LineString":
        return _linestring_to_polygon(geom)

    if geom.geom_type == "MultiLineString":
        polys = []
        for ls in geom.geoms:
            try:
                poly = _linestring_to_polygon(ls)
                polys.append(poly)
            except Exception:
                continue

        if not polys:
            raise ValueError("MultiLineString has no valid polygonal parts")

        if len(polys) == 1:
            return polys[0]
        return MultiPolygon(polys)

    raise ValueError(f"Cannot convert {geom.geom_type} to Polygon")


def _parse_tsv_line(line: str):
    """
    Parse one line of format:

        <id> <WKT> [key#value,key#value,...]

    Returns:
        feat_id, wkt_str, tag_dict
    """
    line = line.rstrip("\n").strip()
    if not line:
        raise ValueError("empty line")

    # First whitespace splits ID from the rest
    first_space = -1
    for ci, ch in enumerate(line):
        if ch in (" ", "\t"):
            first_space = ci
            break

    if first_space == -1:
        raise ValueError("no whitespace found — cannot split ID from WKT")

    feat_id = line[:first_space].strip()

    # Tags start from last '[' if present
    tag_start = line.rfind("[")
    if tag_start != -1:
        tag_raw = line[tag_start:].strip().strip("[]")
        rest = line[first_space:tag_start]
    else:
        tag_raw = ""
        rest = line[first_space:]

    wkt_str = rest.strip()
    if not wkt_str:
        raise ValueError("empty WKT field")

    tag_dict = {}
    for item in tag_raw.split(","):
        item = item.strip()
        if "#" in item:
            k, _, v = item.partition("#")
            tag_dict[k.strip()] = v.strip()

    return feat_id, wkt_str, tag_dict


def load_geometries(path: str) -> Tuple[List, List[str], List[dict]]:
    """
    Load geometries, IDs, and tags.

    Supported:
      - .tsv : <id> <WKT> [tags]
      - .wkt / .txt : one WKT per line
      - others via GeoPandas
    """
    geometries = []
    ids = []
    tags = []
    skipped = 0

    if path.endswith(".tsv"):
        with open(path, encoding="utf-8") as fh:
            for lineno, line in enumerate(fh, 1):
                try:
                    feat_id, wkt_str, tag_dict = _parse_tsv_line(line)
                except ValueError as exc:
                    if line.strip():
                        print(f"[skip] line {lineno}: {exc}")
                    skipped += 1
                    continue

                try:
                    geom = shapely_wkt.loads(wkt_str)
                except Exception as exc:
                    print(f"[skip] line {lineno} id={feat_id}: WKT error — {exc}")
                    print(f"       WKT preview: {wkt_str[:200]}")
                    skipped += 1
                    continue

                try:
                    geom = _geom_to_polygon(geom)
                except Exception as exc:
                    print(f"[skip] line {lineno} id={feat_id}: geometry conversion error — {exc}")
                    print(f"       WKT preview: {wkt_str[:200]}")
                    skipped += 1
                    continue

                if not geom.is_valid:
                    geom = geom.buffer(0)

                if geom.is_empty:
                    skipped += 1
                    continue

                geometries.append(geom)
                ids.append(feat_id)
                tags.append(tag_dict)

    elif path.endswith(".wkt") or path.endswith(".txt"):
        with open(path, encoding="utf-8") as fh:
            for lineno, line in enumerate(fh, 1):
                line = line.strip()
                if not line:
                    continue
                try:
                    geom = shapely_wkt.loads(line)
                    geom = _geom_to_polygon(geom)

                    if not geom.is_valid:
                        geom = geom.buffer(0)

                    if geom.is_empty:
                        raise ValueError("empty geometry after fix")

                    geometries.append(geom)
                    ids.append(str(lineno))
                    tags.append({})
                except Exception as exc:
                    print(f"[skip] line {lineno}: {exc}")
                    skipped += 1

    else:
        import geopandas as gpd

        gdf = gpd.read_file(path)
        for i, row in gdf.iterrows():
            geom = row.geometry
            if geom is None or geom.is_empty:
                skipped += 1
                continue

            try:
                geom = _geom_to_polygon(geom)

                if not geom.is_valid:
                    geom = geom.buffer(0)

                if geom.is_empty:
                    raise ValueError("empty geometry after fix")

                geometries.append(geom)
                ids.append(str(i))
                tags.append({})
            except Exception as exc:
                print(f"[skip] row {i}: {exc}")
                skipped += 1

    print(f"Loaded {len(geometries):,} valid geometries ({skipped} skipped) from {path}")
    return geometries, ids, tags


def to_multipolygon(geom) -> MultiPolygon:
    if geom.geom_type == "Polygon":
        return MultiPolygon([geom])
    elif geom.geom_type == "MultiPolygon":
        return geom
    raise ValueError(f"Unsupported geometry type: {geom.geom_type}")


print("Cell 1 loaded successfully.")

Cell 1 loaded successfully.


In [3]:
# ============================================================
# CELL 2: Silent loader + summary only
# ============================================================

from collections import Counter

DATA_PATH = "/raid/ruban/data/parks.tsv"

def load_geometries_silent(path: str):
    geometries = []
    ids = []
    tags = []

    skipped = 0
    skip_reasons = Counter()

    with open(path, encoding="utf-8") as fh:
        for lineno, line in enumerate(fh, 1):
            try:
                feat_id, wkt_str, tag_dict = _parse_tsv_line(line)
            except Exception:
                skipped += 1
                skip_reasons["parse_error"] += 1
                continue

            try:
                geom = shapely_wkt.loads(wkt_str)
            except Exception:
                skipped += 1
                skip_reasons["wkt_parse_error"] += 1
                continue

            try:
                geom = _geom_to_polygon(geom)
            except Exception as exc:
                skipped += 1
                msg = str(exc).lower()

                if "too short" in msg:
                    skip_reasons["degenerate_linestring"] += 1
                elif "empty" in msg:
                    skip_reasons["empty_geometry"] += 1
                elif "closed ring too short" in msg:
                    skip_reasons["short_closed_ring"] += 1
                else:
                    skip_reasons["geometry_conversion_error"] += 1
                continue

            try:
                if not geom.is_valid:
                    geom = geom.buffer(0)
                if geom.is_empty:
                    skipped += 1
                    skip_reasons["empty_after_fix"] += 1
                    continue
            except Exception:
                skipped += 1
                skip_reasons["post_fix_error"] += 1
                continue

            geometries.append(geom)
            ids.append(feat_id)
            tags.append(tag_dict)

    print("=== LOAD SUMMARY ===")
    print(f"Valid geometries : {len(geometries):,}")
    print(f"Skipped          : {skipped:,}")
    print(f"Kept ratio       : {len(geometries) / (len(geometries) + skipped):.4f}")

    print("\n=== SKIP REASONS ===")
    for k, v in skip_reasons.most_common():
        print(f"{k:24s}: {v:,}")

    return geometries, ids, tags, skip_reasons


# Run this
geometries, ids, tags, skip_reasons = load_geometries_silent(DATA_PATH)

print("\n=== GEOMETRY TYPE SUMMARY ===")
type_counts = Counter(g.geom_type for g in geometries)
for k, v in sorted(type_counts.items()):
    print(f"{k:15s}: {v:,}")

=== LOAD SUMMARY ===
Valid geometries : 234,195
Skipped          : 252
Kept ratio       : 0.9989

=== SKIP REASONS ===
degenerate_linestring   : 196
empty_geometry          : 46
wkt_parse_error         : 10

=== GEOMETRY TYPE SUMMARY ===
MultiPolygon   : 114
Polygon        : 234,081


In [4]:
# ============================================================
# CELL 3: Normalize geometry objects + create a debug subset
# ============================================================

import numpy as np
import random

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

def extract_outer_rings(geom):
    """
    Return a list of outer rings.
    Each ring is a numpy array of shape [num_points, 2].
    """
    rings = []

    if geom.geom_type == "Polygon":
        coords = np.asarray(geom.exterior.coords, dtype=np.float32)
        if len(coords) >= 4:
            rings.append(coords)

    elif geom.geom_type == "MultiPolygon":
        for poly in geom.geoms:
            coords = np.asarray(poly.exterior.coords, dtype=np.float32)
            if len(coords) >= 4:
                rings.append(coords)

    return rings


def ring_signed_area(coords):
    """
    Shoelace signed area for a closed ring.
    """
    x = coords[:, 0]
    y = coords[:, 1]
    return 0.5 * np.sum(x[:-1] * y[1:] - x[1:] * y[:-1])


def choose_main_ring(geom):
    """
    Choose the largest outer ring.
    This keeps the first debug version simple: one ring per geometry.
    """
    rings = extract_outer_rings(geom)
    if not rings:
        return None

    best_ring = None
    best_area = -1.0

    for ring in rings:
        area = abs(ring_signed_area(ring))
        if area > best_area:
            best_area = area
            best_ring = ring

    return best_ring


# Build normalized ring list
main_rings = []
main_ids = []

for geom, feat_id in zip(geometries, ids):
    ring = choose_main_ring(geom)
    if ring is None:
        continue
    main_rings.append(ring)
    main_ids.append(feat_id)

print("=== NORMALIZED DATASET ===")
print(f"Usable geometries with main ring : {len(main_rings):,}")

vertex_counts = np.array([len(r) for r in main_rings], dtype=np.int32)
print(f"Min vertices   : {vertex_counts.min()}")
print(f"Median vertices: {int(np.median(vertex_counts))}")
print(f"p90 vertices   : {int(np.percentile(vertex_counts, 90))}")
print(f"p99 vertices   : {int(np.percentile(vertex_counts, 99))}")
print(f"Max vertices   : {vertex_counts.max()}")

# Small debug subset first
DEBUG_N = 2000   # keep small for now
debug_idx = np.random.choice(len(main_rings), size=min(DEBUG_N, len(main_rings)), replace=False)

rings_debug = [main_rings[i] for i in debug_idx]
ids_debug = [main_ids[i] for i in debug_idx]

print("\n=== DEBUG SUBSET ===")
print(f"Subset size : {len(rings_debug):,}")

for i in range(min(5, len(rings_debug))):
    r = rings_debug[i]
    print(f"[{i}] id={ids_debug[i]} shape={r.shape} first_point={r[0]}")

=== NORMALIZED DATASET ===
Usable geometries with main ring : 234,195
Min vertices   : 4
Median vertices: 10
p90 vertices   : 31
p99 vertices   : 107
Max vertices   : 1895

=== DEBUG SUBSET ===
Subset size : 2,000
[0] id=178469575 shape=(9, 2) first_point=[-68.02319  -38.950733]
[1] id=174443195 shape=(5, 2) first_point=[-73.72178   45.360004]
[2] id=62067021 shape=(19, 2) first_point=[-5.6317344 40.974724 ]
[3] id=77287317 shape=(5, 2) first_point=[-58.505592 -34.515644]
[4] id=37817061 shape=(9, 2) first_point=[ 8.69476  47.254753]


In [5]:
# ============================================================
# CELL 4: Rebuild graph samples with richer node features
# node features:
#   [x, y, prev_edge_len, next_edge_len, turning_cos, turning_sin]
# ============================================================

def normalize_ring(coords):
    coords = np.asarray(coords, dtype=np.float32)

    if len(coords) >= 2 and np.allclose(coords[0], coords[-1]):
        coords = coords[:-1]

    center = coords.mean(axis=0, keepdims=True)
    coords_centered = coords - center

    scale = np.sqrt((coords_centered ** 2).sum(axis=1)).max()
    if scale < 1e-8:
        scale = 1.0

    coords_norm = coords_centered / scale
    return coords_norm.astype(np.float32)


def build_cycle_edge_index(num_nodes):
    src, dst = [], []
    for i in range(num_nodes):
        j = (i + 1) % num_nodes
        src.extend([i, j])
        dst.extend([j, i])
    return np.asarray([src, dst], dtype=np.int64)


def ring_to_rich_node_features(ring):
    """
    Input ring: [N, 2] closed or open ring
    Output features: [N, 6]
    """
    pts = normalize_ring(ring)   # [N, 2], open ring after removing repeated closure
    n = len(pts)

    feats = []

    for i in range(n):
        prev_i = (i - 1) % n
        next_i = (i + 1) % n

        p_prev = pts[prev_i]
        p = pts[i]
        p_next = pts[next_i]

        v_prev = p - p_prev
        v_next = p_next - p

        len_prev = float(np.linalg.norm(v_prev))
        len_next = float(np.linalg.norm(v_next))

        if len_prev < 1e-8:
            u_prev = np.zeros(2, dtype=np.float32)
        else:
            u_prev = v_prev / len_prev

        if len_next < 1e-8:
            u_next = np.zeros(2, dtype=np.float32)
        else:
            u_next = v_next / len_next

        cos_theta = float(np.clip(np.dot(u_prev, u_next), -1.0, 1.0))
        sin_theta = float(u_prev[0] * u_next[1] - u_prev[1] * u_next[0])

        feats.append([
            pts[i, 0],
            pts[i, 1],
            len_prev,
            len_next,
            cos_theta,
            sin_theta,
        ])

    return np.asarray(feats, dtype=np.float32)


def ring_to_graph_sample(ring, feat_id):
    x = ring_to_rich_node_features(ring)          # [N, 6]
    edge_index = build_cycle_edge_index(len(x))   # [2, E]

    return {
        "id": feat_id,
        "x": x,
        "edge_index": edge_index,
        "num_nodes": x.shape[0],
        "num_edges": edge_index.shape[1],
    }


graph_samples_debug = [ring_to_graph_sample(r, fid) for r, fid in zip(rings_debug, ids_debug)]

print("=== RICH GRAPH SAMPLE SUMMARY ===")
print(f"Total graph samples: {len(graph_samples_debug):,}")

g0 = graph_samples_debug[0]
print("id         :", g0["id"])
print("x shape    :", g0["x"].shape)
print("edge shape :", g0["edge_index"].shape)
print("first rows:\n", g0["x"][:5])

=== RICH GRAPH SAMPLE SUMMARY ===
Total graph samples: 2,000
id         : 178469575
x shape    : (8, 6)
edge shape : (2, 16)
first rows:
 [[-0.9428742  -0.3331489   1.8695134   0.5595799  -0.24795279  0.9687722 ]
 [-0.50286627 -0.6788694   0.5595799   0.2828623   0.2583617   0.9660482 ]
 [-0.27657643 -0.50915205  0.2828623   0.30749202 -0.06950369  0.99758166]
 [-0.47772294 -0.27657643  0.30749202  1.0641445  -0.03646681 -0.9993348 ]
 [ 0.35200638  0.38972133  1.0641445   0.29463038 -0.08846754 -0.996079  ]]


In [6]:
# ============================================================
# CELL 5: Convert graph dicts into PyTorch Geometric Data
# ============================================================

import torch

PYG_AVAILABLE = False
pyg_graphs_debug = []

try:
    from torch_geometric.data import Data
    PYG_AVAILABLE = True
    print("torch_geometric is installed.")
except ImportError:
    print("torch_geometric is NOT installed.")


if PYG_AVAILABLE:
    def graph_dict_to_pyg_data(sample):
        x = torch.tensor(sample["x"], dtype=torch.float32)                 # [N, 2]
        edge_index = torch.tensor(sample["edge_index"], dtype=torch.long) # [2, E]

        data = Data(x=x, edge_index=edge_index)
        data.poly_id = str(sample["id"])
        data.num_nodes_manual = sample["num_nodes"]
        data.num_edges_manual = sample["num_edges"]
        return data

    pyg_graphs_debug = [graph_dict_to_pyg_data(g) for g in graph_samples_debug]

    print("\n=== PYG CONVERSION SUMMARY ===")
    print(f"PyG graphs created: {len(pyg_graphs_debug):,}")

    g0 = pyg_graphs_debug[0]
    print("\n=== FIRST PYG GRAPH ===")
    print(g0)
    print("poly_id      :", g0.poly_id)
    print("x.shape      :", tuple(g0.x.shape))
    print("edge_index   :", tuple(g0.edge_index.shape))
    print("num_nodes    :", g0.num_nodes)
    print("first x rows :\n", g0.x[:5])
    print("first edges  :\n", g0.edge_index[:, :10])

else:
    print("\nPyG conversion skipped because torch_geometric is unavailable.")
    print("Next step will be to install torch_geometric or switch to plain PyTorch graph batching.")

torch_geometric is installed.

=== PYG CONVERSION SUMMARY ===
PyG graphs created: 2,000

=== FIRST PYG GRAPH ===
Data(x=[8, 6], edge_index=[2, 16], poly_id='178469575', num_nodes_manual=8, num_edges_manual=16)
poly_id      : 178469575
x.shape      : (8, 6)
edge_index   : (2, 16)
num_nodes    : 8
first x rows :
 tensor([[-0.9429, -0.3331,  1.8695,  0.5596, -0.2480,  0.9688],
        [-0.5029, -0.6789,  0.5596,  0.2829,  0.2584,  0.9660],
        [-0.2766, -0.5092,  0.2829,  0.3075, -0.0695,  0.9976],
        [-0.4777, -0.2766,  0.3075,  1.0641, -0.0365, -0.9993],
        [ 0.3520,  0.3897,  1.0641,  0.2946, -0.0885, -0.9961]])
first edges  :
 tensor([[0, 1, 1, 2, 2, 3, 3, 4, 4, 5],
        [1, 0, 2, 1, 3, 2, 4, 3, 5, 4]])


In [8]:
# ============================================================
# CELL 6: Graph augmentation + train/val split + triplet dataset
#         (fixed for 6D node features)
# ============================================================

import copy
import math
import random
import torch
from torch.utils.data import Dataset

# -----------------------------
# Train / validation split
# -----------------------------
num_graphs = len(pyg_graphs_debug)
indices = list(range(num_graphs))
random.shuffle(indices)

train_ratio = 0.9
train_size = int(train_ratio * num_graphs)

train_idx = indices[:train_size]
val_idx = indices[train_size:]

train_graphs = [pyg_graphs_debug[i] for i in train_idx]
val_graphs = [pyg_graphs_debug[i] for i in val_idx]

print("=== SPLIT SUMMARY ===")
print(f"Total graphs : {num_graphs:,}")
print(f"Train graphs : {len(train_graphs):,}")
print(f"Val graphs   : {len(val_graphs):,}")


# -----------------------------
# Feature rebuild from coords
# -----------------------------
def coords_to_rich_features_torch(coords_xy):
    """
    coords_xy: [N, 2]
    returns:   [N, 6] = [x, y, prev_len, next_len, cos_theta, sin_theta]
    """
    n = coords_xy.shape[0]
    feats = []

    for i in range(n):
        prev_i = (i - 1) % n
        next_i = (i + 1) % n

        p_prev = coords_xy[prev_i]
        p = coords_xy[i]
        p_next = coords_xy[next_i]

        v_prev = p - p_prev
        v_next = p_next - p

        len_prev = torch.norm(v_prev)
        len_next = torch.norm(v_next)

        if float(len_prev) < 1e-8:
            u_prev = torch.zeros(2, dtype=coords_xy.dtype, device=coords_xy.device)
        else:
            u_prev = v_prev / len_prev

        if float(len_next) < 1e-8:
            u_next = torch.zeros(2, dtype=coords_xy.dtype, device=coords_xy.device)
        else:
            u_next = v_next / len_next

        cos_theta = torch.clamp(torch.dot(u_prev, u_next), -1.0, 1.0)
        sin_theta = u_prev[0] * u_next[1] - u_prev[1] * u_next[0]

        feat = torch.stack([
            p[0],
            p[1],
            len_prev,
            len_next,
            cos_theta,
            sin_theta,
        ])
        feats.append(feat)

    return torch.stack(feats, dim=0)


# -----------------------------
# Geometry augmentation
# -----------------------------
def rotate_xy(xy, angle_rad):
    c = math.cos(angle_rad)
    s = math.sin(angle_rad)
    R = torch.tensor([[c, -s], [s, c]], dtype=xy.dtype, device=xy.device)
    return xy @ R.T

def jitter_xy(xy, sigma=0.02):
    return xy + torch.randn_like(xy) * sigma

def scale_xy(xy, scale_min=0.9, scale_max=1.1):
    scale = random.uniform(scale_min, scale_max)
    return xy * scale

def rebuild_cycle_edges(num_nodes, device="cpu"):
    src = []
    dst = []
    for i in range(num_nodes):
        j = (i + 1) % num_nodes
        src.extend([i, j])
        dst.extend([j, i])
    return torch.tensor([src, dst], dtype=torch.long, device=device)

def maybe_reverse_cycle(data):
    if random.random() > 0.5:
        return data

    xy = data.x[:, :2].flip(0)
    x_new = coords_to_rich_features_torch(xy)
    edge_index_new = rebuild_cycle_edges(x_new.shape[0], device=x_new.device)

    out = copy.copy(data)
    out.x = x_new
    out.edge_index = edge_index_new
    return out

def augment_graph(data):
    out = copy.copy(data)

    # only use XY for augmentation
    xy = out.x[:, :2].clone()

    angle = random.uniform(-math.pi / 10, math.pi / 10)
    xy = rotate_xy(xy, angle)
    xy = scale_xy(xy, 0.95, 1.05)
    xy = jitter_xy(xy, sigma=0.01)

    # rebuild full 6D features from transformed XY
    out.x = coords_to_rich_features_torch(xy)
    out.edge_index = rebuild_cycle_edges(out.x.shape[0], device=out.x.device)

    out = maybe_reverse_cycle(out)
    return out


# -----------------------------
# Triplet dataset
# -----------------------------
class PolygonTripletDataset(Dataset):
    def __init__(self, graphs):
        self.graphs = graphs

    def __len__(self):
        return len(self.graphs)

    def __getitem__(self, idx):
        anchor = self.graphs[idx]
        positive = augment_graph(anchor)

        neg_idx = random.randrange(len(self.graphs))
        while neg_idx == idx:
            neg_idx = random.randrange(len(self.graphs))
        negative = self.graphs[neg_idx]

        return anchor, positive, negative


train_triplet_ds = PolygonTripletDataset(train_graphs)
val_triplet_ds = PolygonTripletDataset(val_graphs)

print("\n=== DATASET CHECK ===")
a, p, n = train_triplet_ds[0]
print("Anchor   :", a)
print("Positive :", p)
print("Negative :", n)
print("Anchor x shape   :", tuple(a.x.shape))
print("Positive x shape :", tuple(p.x.shape))
print("Negative x shape :", tuple(n.x.shape))
print("Anchor id        :", a.poly_id)
print("Negative id      :", n.poly_id)

=== SPLIT SUMMARY ===
Total graphs : 2,000
Train graphs : 1,800
Val graphs   : 200

=== DATASET CHECK ===
Anchor   : Data(x=[4, 6], edge_index=[2, 8], poly_id='24652872', num_nodes_manual=4, num_edges_manual=8)
Positive : Data(x=[4, 6], edge_index=[2, 8], poly_id='24652872', num_nodes_manual=4, num_edges_manual=8)
Negative : Data(x=[17, 6], edge_index=[2, 34], poly_id='28545274', num_nodes_manual=17, num_edges_manual=34)
Anchor x shape   : (4, 6)
Positive x shape : (4, 6)
Negative x shape : (17, 6)
Anchor id        : 24652872
Negative id      : 28545274


In [11]:
# ============================================================
# CELL 6B: Hard negative sampling by node-count bucket
# ============================================================

from collections import defaultdict
from torch.utils.data import Dataset

def build_nodecount_buckets(graphs, bucket_width=4):
    """
    Group graph indices by approximate node count.
    Example:
      3-6 nodes -> bucket 0
      7-10 nodes -> bucket 1
      etc.
    """
    buckets = defaultdict(list)

    for i, g in enumerate(graphs):
        n = int(g.x.shape[0])
        bucket_id = n // bucket_width
        buckets[bucket_id].append(i)

    return buckets


train_buckets = build_nodecount_buckets(train_graphs, bucket_width=4)
val_buckets = build_nodecount_buckets(val_graphs, bucket_width=4)

print("=== HARD NEGATIVE BUCKET SUMMARY ===")
print(f"Train buckets: {len(train_buckets)}")
print(f"Val buckets  : {len(val_buckets)}")

# show a few example buckets
shown = 0
for b in sorted(train_buckets.keys()):
    print(f"bucket={b:3d} size={len(train_buckets[b])}")
    shown += 1
    if shown >= 10:
        break


class PolygonTripletDatasetHardNeg(Dataset):
    def __init__(self, graphs, buckets):
        self.graphs = graphs
        self.buckets = buckets

    def __len__(self):
        return len(self.graphs)

    def _sample_hard_negative_index(self, idx):
        anchor = self.graphs[idx]
        n = int(anchor.x.shape[0])
        bucket_id = n // 4

        # try same bucket first
        candidate_pool = [j for j in self.buckets.get(bucket_id, []) if j != idx]

        # if too small, expand to neighboring buckets
        if len(candidate_pool) < 3:
            candidate_pool = []
            for nb in [bucket_id - 1, bucket_id, bucket_id + 1]:
                candidate_pool.extend([j for j in self.buckets.get(nb, []) if j != idx])

        # fallback to random
        if not candidate_pool:
            neg_idx = random.randrange(len(self.graphs))
            while neg_idx == idx:
                neg_idx = random.randrange(len(self.graphs))
            return neg_idx

        return random.choice(candidate_pool)

    def __getitem__(self, idx):
        anchor = self.graphs[idx]
        positive = augment_graph(anchor)

        neg_idx = self._sample_hard_negative_index(idx)
        negative = self.graphs[neg_idx]

        return anchor, positive, negative


# overwrite datasets
train_triplet_ds = PolygonTripletDatasetHardNeg(train_graphs, train_buckets)
val_triplet_ds = PolygonTripletDatasetHardNeg(val_graphs, val_buckets)

print("\n=== HARD NEGATIVE DATASET CHECK ===")
a, p, n = train_triplet_ds[0]
print("Anchor nodes   :", a.x.shape[0], "id =", a.poly_id)
print("Positive nodes :", p.x.shape[0], "id =", p.poly_id)
print("Negative nodes :", n.x.shape[0], "id =", n.poly_id)

=== HARD NEGATIVE BUCKET SUMMARY ===
Train buckets: 35
Val buckets  : 18
bucket=  0 size=28
bucket=  1 size=784
bucket=  2 size=314
bucket=  3 size=189
bucket=  4 size=130
bucket=  5 size=88
bucket=  6 size=57
bucket=  7 size=35
bucket=  8 size=26
bucket=  9 size=37

=== HARD NEGATIVE DATASET CHECK ===
Anchor nodes   : 4 id = 24652872
Positive nodes : 4 id = 24652872
Negative nodes : 4 id = 50250590


In [16]:
# ============================================================
# CELL 6C: Hybrid negatives (hard + random)
# ============================================================

class PolygonTripletDatasetHybrid(Dataset):
    def __init__(self, graphs, buckets):
        self.graphs = graphs
        self.buckets = buckets

    def __len__(self):
        return len(self.graphs)

    def _sample_hard_negative(self, idx):
        anchor = self.graphs[idx]
        n = int(anchor.x.shape[0])
        bucket_id = n // 4

        candidates = [j for j in self.buckets.get(bucket_id, []) if j != idx]

        if not candidates:
            return None

        return random.choice(candidates)

    def _sample_random_negative(self, idx):
        neg_idx = random.randrange(len(self.graphs))
        while neg_idx == idx:
            neg_idx = random.randrange(len(self.graphs))
        return neg_idx

    def __getitem__(self, idx):
        anchor = self.graphs[idx]
        positive = augment_graph(anchor)

        # mix: 50% hard, 50% random
        if random.random() < 0.5:
            neg_idx = self._sample_hard_negative(idx)
            if neg_idx is None:
                neg_idx = self._sample_random_negative(idx)
        else:
            neg_idx = self._sample_random_negative(idx)

        negative = self.graphs[neg_idx]

        return anchor, positive, negative


# overwrite datasets
train_triplet_ds = PolygonTripletDatasetHybrid(train_graphs, train_buckets)
val_triplet_ds = PolygonTripletDatasetHybrid(val_graphs, val_buckets)

print("Hybrid dataset ready")

Hybrid dataset ready


In [17]:
# ============================================================
# CELL 7: PolygonGNN model
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.nn import GCNConv, global_mean_pool

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)


class PolygonGNN(nn.Module):
    def __init__(self, in_dim=2, hidden_dim=64, emb_dim=128, dropout=0.1):
        super().__init__()

        self.conv1 = GCNConv(in_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.conv3 = GCNConv(hidden_dim, hidden_dim)

        self.dropout = nn.Dropout(dropout)

        self.proj = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, emb_dim),
        )

    def encode(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.dropout(x)

        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = self.dropout(x)

        x = self.conv3(x, edge_index)
        x = F.relu(x)

        g = global_mean_pool(x, batch)   # [num_graphs, hidden_dim]
        z = self.proj(g)                 # [num_graphs, emb_dim]
        z = F.normalize(z, p=2, dim=1)   # normalized embedding
        return z

    def forward(self, data):
        return self.encode(data)


model = PolygonGNN(
    in_dim=6,
    hidden_dim=64,
    emb_dim=128,
    dropout=0.10
).to(DEVICE)

print(model)

# Quick smoke test with one graph
from torch_geometric.loader import DataLoader as PyGDataLoader

tmp_loader = PyGDataLoader(train_graphs[:4], batch_size=4, shuffle=False)
tmp_batch = next(iter(tmp_loader)).to(DEVICE)

with torch.no_grad():
    tmp_z = model(tmp_batch)

print("\n=== MODEL SMOKE TEST ===")
print("Batch object :", tmp_batch)
print("Embedding shape:", tuple(tmp_z.shape))
print("First embedding norm:", float(tmp_z[0].norm()))

Using device: cuda
PolygonGNN(
  (conv1): GCNConv(6, 64)
  (conv2): GCNConv(64, 64)
  (conv3): GCNConv(64, 64)
  (dropout): Dropout(p=0.1, inplace=False)
  (proj): Sequential(
    (0): Linear(in_features=64, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
    (3): Linear(in_features=64, out_features=128, bias=True)
  )
)

=== MODEL SMOKE TEST ===
Batch object : DataBatch(x=[25, 6], edge_index=[2, 50], poly_id=[4], num_nodes_manual=[4], num_edges_manual=[4], batch=[25], ptr=[5])
Embedding shape: (4, 128)
First embedding norm: 0.9999999403953552


In [18]:
# ============================================================
# CELL 8: Triplet training loop
# ============================================================

import torch
import torch.nn.functional as F
from torch_geometric.loader import DataLoader as PyGDataLoader

# -----------------------------
# Collate triplets into 3 loaders
# -----------------------------
def make_triplet_batches(dataset, batch_size=64, shuffle=True):
    indices = list(range(len(dataset)))
    if shuffle:
        random.shuffle(indices)

    for start in range(0, len(indices), batch_size):
        batch_idx = indices[start:start + batch_size]

        anchors = []
        positives = []
        negatives = []

        for idx in batch_idx:
            a, p, n = dataset[idx]
            anchors.append(a)
            positives.append(p)
            negatives.append(n)

        yield (
            PyGDataLoader(anchors, batch_size=len(anchors), shuffle=False),
            PyGDataLoader(positives, batch_size=len(positives), shuffle=False),
            PyGDataLoader(negatives, batch_size=len(negatives), shuffle=False),
        )


def get_single_batch(loader):
    return next(iter(loader))


triplet_loss_fn = torch.nn.TripletMarginLoss(margin=0.20, p=2)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

EPOCHS = 8
BATCH_SIZE = 64

train_history = []
val_history = []

for epoch in range(1, EPOCHS + 1):
    # -------------------------
    # Train
    # -------------------------
    model.train()
    train_losses = []

    for a_loader, p_loader, n_loader in make_triplet_batches(
        train_triplet_ds, batch_size=BATCH_SIZE, shuffle=True
    ):
        a_batch = get_single_batch(a_loader).to(DEVICE)
        p_batch = get_single_batch(p_loader).to(DEVICE)
        n_batch = get_single_batch(n_loader).to(DEVICE)

        z_a = model(a_batch)
        z_p = model(p_batch)
        z_n = model(n_batch)

        loss = triplet_loss_fn(z_a, z_p, z_n)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())

    train_loss = float(np.mean(train_losses)) if train_losses else float("nan")
    train_history.append(train_loss)

    # -------------------------
    # Validation
    # -------------------------
    model.eval()
    val_losses = []

    with torch.no_grad():
        for a_loader, p_loader, n_loader in make_triplet_batches(
            val_triplet_ds, batch_size=BATCH_SIZE, shuffle=False
        ):
            a_batch = get_single_batch(a_loader).to(DEVICE)
            p_batch = get_single_batch(p_loader).to(DEVICE)
            n_batch = get_single_batch(n_loader).to(DEVICE)

            z_a = model(a_batch)
            z_p = model(p_batch)
            z_n = model(n_batch)

            loss = triplet_loss_fn(z_a, z_p, z_n)
            val_losses.append(loss.item())

    val_loss = float(np.mean(val_losses)) if val_losses else float("nan")
    val_history.append(val_loss)

    print(f"Epoch {epoch:02d} | train_loss={train_loss:.6f} | val_loss={val_loss:.6f}")

Epoch 01 | train_loss=0.118776 | val_loss=0.059887
Epoch 02 | train_loss=0.073689 | val_loss=0.052894
Epoch 03 | train_loss=0.056043 | val_loss=0.028864
Epoch 04 | train_loss=0.052654 | val_loss=0.022102
Epoch 05 | train_loss=0.049507 | val_loss=0.019672
Epoch 06 | train_loss=0.043745 | val_loss=0.030856
Epoch 07 | train_loss=0.043227 | val_loss=0.019990
Epoch 08 | train_loss=0.047495 | val_loss=0.031280


In [19]:
# ============================================================
# CELL 9: Retrieval evaluation vs simple baseline
# ============================================================

from torch_geometric.loader import DataLoader as PyGDataLoader
import numpy as np

# -----------------------------
# 1) Extract GNN embeddings
# -----------------------------
model.eval()

embed_loader = PyGDataLoader(pyg_graphs_debug, batch_size=128, shuffle=False)

all_embeds = []
all_ids = []
all_num_nodes = []

with torch.no_grad():
    for batch in embed_loader:
        batch = batch.to(DEVICE)
        z = model(batch)
        all_embeds.append(z.cpu().numpy())
        all_ids.extend(batch.poly_id)
        all_num_nodes.extend([int(v) for v in batch.num_nodes_manual])

embeddings = np.vstack(all_embeds).astype(np.float32)
all_num_nodes = np.asarray(all_num_nodes, dtype=np.int32)

print("=== EMBEDDING SUMMARY ===")
print("embeddings shape:", embeddings.shape)
print("num ids         :", len(all_ids))

# cosine similarity because embeddings are normalized
sim_gnn = embeddings @ embeddings.T
np.fill_diagonal(sim_gnn, -1.0)

# -----------------------------
# 2) Node-count baseline
# -----------------------------
# similarity = negative absolute difference in node count
node_diff = np.abs(all_num_nodes[:, None] - all_num_nodes[None, :]).astype(np.float32)
sim_count = -node_diff
np.fill_diagonal(sim_count, -1e9)

# -----------------------------
# 3) Build a weak geometric proxy ground truth
#    based on ring descriptors
# -----------------------------
def ring_descriptor(ring):
    """
    Simple handcrafted descriptor for quick evaluation.
    Not perfect ground truth, but much better than nothing.
    """
    pts = normalize_ring(ring)  # [N,2], open ring
    n = len(pts)

    # edge lengths
    nxt = np.roll(pts, -1, axis=0)
    edges = nxt - pts
    lens = np.linalg.norm(edges, axis=1)
    lens = np.sort(lens)

    # angle cosines
    angle_cos = []
    for i in range(n):
        p_prev = pts[(i - 1) % n]
        p = pts[i]
        p_next = pts[(i + 1) % n]

        v1 = p - p_prev
        v2 = p_next - p

        nv1 = np.linalg.norm(v1)
        nv2 = np.linalg.norm(v2)

        if nv1 < 1e-8 or nv2 < 1e-8:
            c = 1.0
        else:
            c = np.clip(np.dot(v1 / nv1, v2 / nv2), -1.0, 1.0)
        angle_cos.append(c)

    angle_cos = np.sort(np.asarray(angle_cos, dtype=np.float32))

    # fixed-length summary
    def summarize(arr):
        return np.array([
            arr.min(),
            np.median(arr),
            np.percentile(arr, 75),
            arr.max()
        ], dtype=np.float32)

    desc = np.concatenate([
        np.array([n], dtype=np.float32),
        summarize(lens),
        summarize(angle_cos),
    ])
    return desc

ring_desc = np.vstack([ring_descriptor(r) for r in rings_debug]).astype(np.float32)

# pairwise L2 distance in descriptor space
# smaller = more similar
desc_sq = np.sum(ring_desc ** 2, axis=1, keepdims=True)
desc_d2 = desc_sq + desc_sq.T - 2.0 * (ring_desc @ ring_desc.T)
desc_d2 = np.maximum(desc_d2, 0.0)
np.fill_diagonal(desc_d2, np.inf)

# ground-truth neighbors from descriptor distance
GT_TOPK = 10
gt_neighbors = np.argsort(desc_d2, axis=1)[:, :GT_TOPK]

print("\nDescriptor shape:", ring_desc.shape)
print("GT neighbor matrix shape:", gt_neighbors.shape)

# -----------------------------
# 4) Evaluate recall@K
# -----------------------------
def recall_at_k(pred_neighbors, gt_neighbors, k):
    recalls = []
    for i in range(len(pred_neighbors)):
        gt_set = set(gt_neighbors[i][:k].tolist())
        pred_set = set(pred_neighbors[i][:k].tolist())
        if len(gt_set) == 0:
            continue
        recalls.append(len(gt_set & pred_set) / len(gt_set))
    return float(np.mean(recalls))

pred_gnn_5  = np.argsort(-sim_gnn, axis=1)[:, :5]
pred_gnn_10 = np.argsort(-sim_gnn, axis=1)[:, :10]

pred_cnt_5  = np.argsort(-sim_count, axis=1)[:, :5]
pred_cnt_10 = np.argsort(-sim_count, axis=1)[:, :10]

r_gnn_5  = recall_at_k(pred_gnn_5,  gt_neighbors, 5)
r_gnn_10 = recall_at_k(pred_gnn_10, gt_neighbors, 10)

r_cnt_5  = recall_at_k(pred_cnt_5,  gt_neighbors, 5)
r_cnt_10 = recall_at_k(pred_cnt_10, gt_neighbors, 10)

print("\n=== RETRIEVAL EVALUATION ===")
print(f"Node-count baseline Recall@5  : {r_cnt_5:.4f}")
print(f"Node-count baseline Recall@10 : {r_cnt_10:.4f}")
print(f"GNN embedding Recall@5        : {r_gnn_5:.4f}")
print(f"GNN embedding Recall@10       : {r_gnn_10:.4f}")

=== EMBEDDING SUMMARY ===
embeddings shape: (2000, 128)
num ids         : 2000

Descriptor shape: (2000, 9)
GT neighbor matrix shape: (2000, 10)

=== RETRIEVAL EVALUATION ===
Node-count baseline Recall@5  : 0.1470
Node-count baseline Recall@10 : 0.2237
GNN embedding Recall@5        : 0.0706
GNN embedding Recall@10       : 0.0995


In [23]:
# ============================================================
# CELL 11: Load real GT queries and build GT-focused subset
# ============================================================

import os
import numpy as np

GT_DIR = "/raid/ruban/groundtruth/pk-query-187019"

NUM_GT_QUERIES = 300          # start moderate
GT_TOPK_PER_QUERY = 20        # use top-20 neighbors from GT

def load_gt_subset(gt_dir, max_queries=300, topk_per_query=20):
    """
    Reads GT shard text files of format:
      query_idx, nn1, nn2, nn3, ...
    Returns:
      gt_map: dict[int, list[int]]
    """
    gt_map = {}

    shard_files = sorted(os.listdir(gt_dir))
    for fname in shard_files:
        fpath = os.path.join(gt_dir, fname)

        with open(fpath, "r", encoding="utf-8", errors="ignore") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue

                parts = [p.strip() for p in line.split(",") if p.strip()]
                vals = [int(x) for x in parts]

                qid = vals[0]
                neigh = vals[1:1 + topk_per_query]

                gt_map[qid] = neigh

                if len(gt_map) >= max_queries:
                    return gt_map

    return gt_map


gt_real = load_gt_subset(
    GT_DIR,
    max_queries=NUM_GT_QUERIES,
    topk_per_query=GT_TOPK_PER_QUERY
)

print("=== REAL GT SUBSET ===")
print("Loaded GT queries:", len(gt_real))

sample_qids = list(gt_real.keys())[:5]
print("Sample query indices:", sample_qids)

for q in sample_qids[:3]:
    print(f"q={q} -> top neighbors: {gt_real[q][:10]}")


# ------------------------------------------------------------
# Build induced subset = queries + their GT neighbors
# ------------------------------------------------------------
focus_global_indices = set()

for q, neighs in gt_real.items():
    if 0 <= q < len(main_rings):
        focus_global_indices.add(q)
    for n in neighs:
        if 0 <= n < len(main_rings):
            focus_global_indices.add(n)

focus_global_indices = sorted(focus_global_indices)

print("\n=== FOCUSED SUBSET ===")
print("Focused subset size:", len(focus_global_indices))
print("Min global idx     :", min(focus_global_indices))
print("Max global idx     :", max(focus_global_indices))

# global -> local mapping for focused subset
g2l = {g: i for i, g in enumerate(focus_global_indices)}
l2g = {i: g for g, i in g2l.items()}

# materialize focused rings / geometries
rings_focus = [main_rings[g] for g in focus_global_indices]
geoms_focus = [geometries[g] for g in focus_global_indices]

print("\n=== SANITY ===")
print("rings_focus:", len(rings_focus))
print("geoms_focus:", len(geoms_focus))
print("Sample focused globals:", focus_global_indices[:10])

=== REAL GT SUBSET ===
Loaded GT queries: 300
Sample query indices: [187019, 187020, 187021, 187022, 187023]
q=187019 -> top neighbors: [184495, 105197, 51737, 107491, 130402, 79996, 20940, 138150, 135921, 123708]
q=187020 -> top neighbors: [173862, 155387, 20083, 144827, 118043, 85940, 165704, 28890, 166989, 28759]
q=187021 -> top neighbors: [181444, 106463, 174581, 106307, 102414, 86564, 27329, 87687, 3868, 22553]

=== FOCUSED SUBSET ===
Focused subset size: 5775
Min global idx     : 57
Max global idx     : 187318

=== SANITY ===
rings_focus: 5775
geoms_focus: 5775
Sample focused globals: [57, 136, 140, 163, 191, 251, 260, 311, 443, 469]


In [24]:
# ============================================================
# CELL 12: Build focused graph set + local GT mapping + query split
# ============================================================

import random
import numpy as np
import torch
from torch_geometric.data import Data

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ------------------------------------------------------------
# 1) Rebuild graph samples for focused subset
#    Uses your current rich-feature ring_to_graph_sample(...)
# ------------------------------------------------------------
graph_samples_focus = []
for gidx in focus_global_indices:
    ring = main_rings[gidx]
    sample = ring_to_graph_sample(ring, feat_id=str(gidx))   # use global cleaned index as ID
    sample["global_idx"] = gidx
    graph_samples_focus.append(sample)

print("=== FOCUSED GRAPH SAMPLES ===")
print("Total graph samples:", len(graph_samples_focus))
print("First sample keys  :", list(graph_samples_focus[0].keys()))
print("First sample id    :", graph_samples_focus[0]["id"])
print("First x shape      :", graph_samples_focus[0]["x"].shape)
print("First edge shape   :", graph_samples_focus[0]["edge_index"].shape)


# ------------------------------------------------------------
# 2) Convert to PyG
# ------------------------------------------------------------
def graph_dict_to_pyg_data_focus(sample):
    x = torch.tensor(sample["x"], dtype=torch.float32)
    edge_index = torch.tensor(sample["edge_index"], dtype=torch.long)

    data = Data(x=x, edge_index=edge_index)
    data.poly_id = str(sample["id"])                 # global cleaned index as string
    data.global_idx = int(sample["global_idx"])
    data.num_nodes_manual = int(sample["num_nodes"])
    data.num_edges_manual = int(sample["num_edges"])
    return data

pyg_graphs_focus = [graph_dict_to_pyg_data_focus(g) for g in graph_samples_focus]

print("\n=== FOCUSED PYG SUMMARY ===")
print("PyG graphs created:", len(pyg_graphs_focus))
print("First PyG graph   :", pyg_graphs_focus[0])


# ------------------------------------------------------------
# 3) Build local GT map
#    gt_real is global-index based; convert to local-index based
# ------------------------------------------------------------
gt_local = {}

for q_global, neigh_globals in gt_real.items():
    if q_global not in g2l:
        continue

    q_local = g2l[q_global]
    neigh_local = [g2l[n] for n in neigh_globals if n in g2l]

    if len(neigh_local) > 0:
        gt_local[q_local] = neigh_local

print("\n=== LOCAL GT MAP ===")
print("Queries with local GT:", len(gt_local))

sample_local_qs = list(gt_local.keys())[:5]
print("Sample local query ids:", sample_local_qs)

for ql in sample_local_qs[:3]:
    print(f"local_q={ql} global_q={l2g[ql]} -> local_neighbors={gt_local[ql][:10]}")


# ------------------------------------------------------------
# 4) Train/val split on QUERY set only
# ------------------------------------------------------------
query_locals = sorted(gt_local.keys())
random.shuffle(query_locals)

train_ratio = 0.8
train_q_count = int(train_ratio * len(query_locals))

train_query_locals = query_locals[:train_q_count]
val_query_locals = query_locals[train_q_count:]

print("\n=== QUERY SPLIT ===")
print("Total GT queries :", len(query_locals))
print("Train queries    :", len(train_query_locals))
print("Val queries      :", len(val_query_locals))

print("\nSample train globals:", [l2g[q] for q in train_query_locals[:5]])
print("Sample val globals  :", [l2g[q] for q in val_query_locals[:5]])

=== FOCUSED GRAPH SAMPLES ===
Total graph samples: 5775
First sample keys  : ['id', 'x', 'edge_index', 'num_nodes', 'num_edges', 'global_idx']
First sample id    : 57
First x shape      : (4, 6)
First edge shape   : (2, 8)

=== FOCUSED PYG SUMMARY ===
PyG graphs created: 5775
First PyG graph   : Data(x=[4, 6], edge_index=[2, 8], poly_id='57', global_idx=57, num_nodes_manual=4, num_edges_manual=8)

=== LOCAL GT MAP ===
Queries with local GT: 289
Sample local query ids: [5475, 5476, 5477, 5478, 5479]
local_q=5475 global_q=187019 -> local_neighbors=[5412, 3058, 1488, 3138, 3809, 2274, 610, 4026, 3974, 3611]
local_q=5476 global_q=187020 -> local_neighbors=[5086, 4542, 575, 4222, 3432, 2455, 4846, 827, 4890, 823]
local_q=5477 global_q=187021 -> local_neighbors=[5316, 3101, 5104, 3095, 2978, 2473, 785, 2506, 104, 652]

=== QUERY SPLIT ===
Total GT queries : 289
Train queries    : 231
Val queries      : 58

Sample train globals: [187309, 187080, 187221, 187288, 187229]
Sample val globals  : [

In [25]:
# ============================================================
# CELL 13: Real GT-supervised triplet dataset
# anchor   = GT query
# positive = true Jaccard neighbor from gt_local
# negative = non-neighbor sampled from focused subset
# ============================================================

import copy
import math
import random
import torch
from torch.utils.data import Dataset

# ------------------------------------------------------------
# Optional mild augmentation for positives/anchors
# Keep it very small so we do not destroy GT semantics
# ------------------------------------------------------------
def coords_to_rich_features_torch(coords_xy):
    n = coords_xy.shape[0]
    feats = []

    for i in range(n):
        prev_i = (i - 1) % n
        next_i = (i + 1) % n

        p_prev = coords_xy[prev_i]
        p = coords_xy[i]
        p_next = coords_xy[next_i]

        v_prev = p - p_prev
        v_next = p_next - p

        len_prev = torch.norm(v_prev)
        len_next = torch.norm(v_next)

        if float(len_prev) < 1e-8:
            u_prev = torch.zeros(2, dtype=coords_xy.dtype, device=coords_xy.device)
        else:
            u_prev = v_prev / len_prev

        if float(len_next) < 1e-8:
            u_next = torch.zeros(2, dtype=coords_xy.dtype, device=coords_xy.device)
        else:
            u_next = v_next / len_next

        cos_theta = torch.clamp(torch.dot(u_prev, u_next), -1.0, 1.0)
        sin_theta = u_prev[0] * u_next[1] - u_prev[1] * u_next[0]

        feats.append(torch.stack([
            p[0], p[1], len_prev, len_next, cos_theta, sin_theta
        ]))

    return torch.stack(feats, dim=0)

def rotate_xy(xy, angle_rad):
    c = math.cos(angle_rad)
    s = math.sin(angle_rad)
    R = torch.tensor([[c, -s], [s, c]], dtype=xy.dtype, device=xy.device)
    return xy @ R.T

def jitter_xy(xy, sigma=0.005):
    return xy + torch.randn_like(xy) * sigma

def scale_xy(xy, scale_min=0.98, scale_max=1.02):
    scale = random.uniform(scale_min, scale_max)
    return xy * scale

def rebuild_cycle_edges(num_nodes, device="cpu"):
    src = []
    dst = []
    for i in range(num_nodes):
        j = (i + 1) % num_nodes
        src.extend([i, j])
        dst.extend([j, i])
    return torch.tensor([src, dst], dtype=torch.long, device=device)

def mild_augment_graph(data, p_apply=0.5):
    if random.random() > p_apply:
        return data

    out = copy.copy(data)
    xy = out.x[:, :2].clone()

    angle = random.uniform(-math.pi / 20, math.pi / 20)
    xy = rotate_xy(xy, angle)
    xy = scale_xy(xy, 0.99, 1.01)
    xy = jitter_xy(xy, sigma=0.003)

    out.x = coords_to_rich_features_torch(xy)
    out.edge_index = rebuild_cycle_edges(out.x.shape[0], device=out.x.device)
    return out


# ------------------------------------------------------------
# Negative buckets by node count for harder negatives
# ------------------------------------------------------------
from collections import defaultdict

def build_nodecount_buckets(graphs, bucket_width=4):
    buckets = defaultdict(list)
    for i, g in enumerate(graphs):
        n = int(g.x.shape[0])
        bucket_id = n // bucket_width
        buckets[bucket_id].append(i)
    return buckets

focus_buckets = build_nodecount_buckets(pyg_graphs_focus, bucket_width=4)

print("=== FOCUS BUCKET SUMMARY ===")
print("Num buckets:", len(focus_buckets))


# ------------------------------------------------------------
# GT-supervised dataset
# ------------------------------------------------------------
class RealGTTripletDataset(Dataset):
    def __init__(self, graphs, gt_local, query_local_ids, buckets):
        self.graphs = graphs
        self.gt_local = gt_local
        self.query_local_ids = query_local_ids
        self.buckets = buckets

    def __len__(self):
        return len(self.query_local_ids)

    def _sample_positive(self, q_local):
        pos_candidates = self.gt_local[q_local]
        return random.choice(pos_candidates)

    def _sample_negative(self, q_local):
        anchor = self.graphs[q_local]
        anchor_nodes = int(anchor.x.shape[0])
        bucket_id = anchor_nodes // 4

        forbidden = set(self.gt_local[q_local])
        forbidden.add(q_local)

        # same bucket first
        pool = [j for j in self.buckets.get(bucket_id, []) if j not in forbidden]

        # expand if needed
        if len(pool) < 10:
            pool = []
            for nb in [bucket_id - 1, bucket_id, bucket_id + 1]:
                pool.extend([j for j in self.buckets.get(nb, []) if j not in forbidden])

        # fallback global
        if not pool:
            pool = [j for j in range(len(self.graphs)) if j not in forbidden]

        return random.choice(pool)

    def __getitem__(self, idx):
        q_local = self.query_local_ids[idx]

        pos_local = self._sample_positive(q_local)
        neg_local = self._sample_negative(q_local)

        anchor = self.graphs[q_local]
        positive = self.graphs[pos_local]
        negative = self.graphs[neg_local]

        # mild augmentation only
        anchor = mild_augment_graph(anchor, p_apply=0.3)
        positive = mild_augment_graph(positive, p_apply=0.3)

        return anchor, positive, negative


train_triplet_ds = RealGTTripletDataset(
    graphs=pyg_graphs_focus,
    gt_local=gt_local,
    query_local_ids=train_query_locals,
    buckets=focus_buckets
)

val_triplet_ds = RealGTTripletDataset(
    graphs=pyg_graphs_focus,
    gt_local=gt_local,
    query_local_ids=val_query_locals,
    buckets=focus_buckets
)

print("\n=== GT DATASET CHECK ===")
a, p, n = train_triplet_ds[0]
print("Anchor   :", a)
print("Positive :", p)
print("Negative :", n)
print("Anchor global   :", a.global_idx)
print("Positive global :", p.global_idx)
print("Negative global :", n.global_idx)
print("Anchor nodes    :", a.x.shape[0])
print("Positive nodes  :", p.x.shape[0])
print("Negative nodes  :", n.x.shape[0])

=== FOCUS BUCKET SUMMARY ===
Num buckets: 55

=== GT DATASET CHECK ===
Anchor   : Data(x=[4, 6], edge_index=[2, 8], poly_id='187309', global_idx=187309, num_nodes_manual=4, num_edges_manual=8)
Positive : Data(x=[6, 6], edge_index=[2, 12], poly_id='71963', global_idx=71963, num_nodes_manual=6, num_edges_manual=12)
Negative : Data(x=[4, 6], edge_index=[2, 8], poly_id='136009', global_idx=136009, num_nodes_manual=4, num_edges_manual=8)
Anchor global   : 187309
Positive global : 71963
Negative global : 136009
Anchor nodes    : 4
Positive nodes  : 6
Negative nodes  : 4


In [26]:
# ============================================================
# CELL 14: Train on REAL GT triplets
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random

from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.loader import DataLoader as PyGDataLoader

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# -----------------------------
# Model
# -----------------------------
class PolygonGNN(nn.Module):
    def __init__(self, in_dim=6, hidden_dim=64, emb_dim=128, dropout=0.10):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.conv3 = GCNConv(hidden_dim, hidden_dim)

        self.dropout = nn.Dropout(dropout)

        self.proj = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, emb_dim),
        )

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.dropout(x)

        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = self.dropout(x)

        x = self.conv3(x, edge_index)
        x = F.relu(x)

        g = global_mean_pool(x, batch)
        z = self.proj(g)
        z = F.normalize(z, p=2, dim=1)
        return z


model = PolygonGNN(in_dim=6, hidden_dim=64, emb_dim=128, dropout=0.10).to(DEVICE)
print(model)

# -----------------------------
# Batch helper
# -----------------------------
def make_triplet_batches(dataset, batch_size=48, shuffle=True):
    indices = list(range(len(dataset)))
    if shuffle:
        random.shuffle(indices)

    for start in range(0, len(indices), batch_size):
        batch_idx = indices[start:start + batch_size]

        anchors, positives, negatives = [], [], []
        for idx in batch_idx:
            a, p, n = dataset[idx]
            anchors.append(a)
            positives.append(p)
            negatives.append(n)

        yield (
            next(iter(PyGDataLoader(anchors, batch_size=len(anchors), shuffle=False))),
            next(iter(PyGDataLoader(positives, batch_size=len(positives), shuffle=False))),
            next(iter(PyGDataLoader(negatives, batch_size=len(negatives), shuffle=False))),
        )

# -----------------------------
# Train
# -----------------------------
triplet_loss_fn = nn.TripletMarginLoss(margin=0.20, p=2)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

EPOCHS = 12
BATCH_SIZE = 48

train_history = []
val_history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_losses = []

    for a_batch, p_batch, n_batch in make_triplet_batches(
        train_triplet_ds, batch_size=BATCH_SIZE, shuffle=True
    ):
        a_batch = a_batch.to(DEVICE)
        p_batch = p_batch.to(DEVICE)
        n_batch = n_batch.to(DEVICE)

        z_a = model(a_batch)
        z_p = model(p_batch)
        z_n = model(n_batch)

        loss = triplet_loss_fn(z_a, z_p, z_n)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())

    train_loss = float(np.mean(train_losses))
    train_history.append(train_loss)

    model.eval()
    val_losses = []

    with torch.no_grad():
        for a_batch, p_batch, n_batch in make_triplet_batches(
            val_triplet_ds, batch_size=BATCH_SIZE, shuffle=False
        ):
            a_batch = a_batch.to(DEVICE)
            p_batch = p_batch.to(DEVICE)
            n_batch = n_batch.to(DEVICE)

            z_a = model(a_batch)
            z_p = model(p_batch)
            z_n = model(n_batch)

            loss = triplet_loss_fn(z_a, z_p, z_n)
            val_losses.append(loss.item())

    val_loss = float(np.mean(val_losses))
    val_history.append(val_loss)

    print(f"Epoch {epoch:02d} | train_loss={train_loss:.6f} | val_loss={val_loss:.6f}")

Using device: cuda
PolygonGNN(
  (conv1): GCNConv(6, 64)
  (conv2): GCNConv(64, 64)
  (conv3): GCNConv(64, 64)
  (dropout): Dropout(p=0.1, inplace=False)
  (proj): Sequential(
    (0): Linear(in_features=64, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
    (3): Linear(in_features=64, out_features=128, bias=True)
  )
)
Epoch 01 | train_loss=0.220900 | val_loss=0.203550
Epoch 02 | train_loss=0.201398 | val_loss=0.218991
Epoch 03 | train_loss=0.206779 | val_loss=0.212879
Epoch 04 | train_loss=0.202836 | val_loss=0.211401
Epoch 05 | train_loss=0.194332 | val_loss=0.209470
Epoch 06 | train_loss=0.197605 | val_loss=0.212036
Epoch 07 | train_loss=0.198614 | val_loss=0.217531
Epoch 08 | train_loss=0.195842 | val_loss=0.200656
Epoch 09 | train_loss=0.197291 | val_loss=0.216326
Epoch 10 | train_loss=0.207365 | val_loss=0.201461
Epoch 11 | train_loss=0.198797 | val_loss=0.199839
Epoch 12 | train_loss=0.193333 | val_loss=0.199499


In [28]:
# ============================================================
# CELL 15: REAL GT retrieval evaluation
# ============================================================

import numpy as np
from torch_geometric.loader import DataLoader as PyGDataLoader

model.eval()

# -----------------------------
# 1) Embed all focused graphs
# -----------------------------
embed_loader = PyGDataLoader(pyg_graphs_focus, batch_size=128, shuffle=False)

focus_embeds = []
focus_globals = []

with torch.no_grad():
    for batch in embed_loader:
        batch = batch.to(DEVICE)
        z = model(batch)
        focus_embeds.append(z.cpu().numpy())
        focus_globals.extend([int(x) for x in batch.global_idx])

focus_embeddings = np.vstack(focus_embeds).astype(np.float32)
focus_globals = np.asarray(focus_globals, dtype=np.int32)

print("=== FOCUS EMBEDDING SUMMARY ===")
print("focus_embeddings shape:", focus_embeddings.shape)
print("num focused graphs    :", len(focus_globals))

# cosine similarity because embeddings are normalized
sim = focus_embeddings @ focus_embeddings.T
np.fill_diagonal(sim, -1.0)

# -----------------------------
# 2) Baseline: node-count similarity
# -----------------------------
focus_num_nodes = np.asarray([int(g.num_nodes_manual) for g in pyg_graphs_focus], dtype=np.int32)
node_diff = np.abs(focus_num_nodes[:, None] - focus_num_nodes[None, :]).astype(np.float32)
sim_count = -node_diff
np.fill_diagonal(sim_count, -1e9)

# -----------------------------
# 3) Recall helpers
# -----------------------------
def recall_at_k_for_queries(sim_matrix, gt_local_map, query_local_ids, k=10):
    recalls = []
    pred_neighbors = np.argsort(-sim_matrix, axis=1)[:, :k]

    for q in query_local_ids:
        if q not in gt_local_map:
            continue

        gt_set = set(gt_local_map[q][:k])
        pred_set = set(pred_neighbors[q][:k])

        if len(gt_set) == 0:
            continue

        recalls.append(len(gt_set & pred_set) / len(gt_set))

    return float(np.mean(recalls)) if recalls else 0.0

def hitrate_at_k_for_queries(sim_matrix, gt_local_map, query_local_ids, k=10):
    hits = []
    pred_neighbors = np.argsort(-sim_matrix, axis=1)[:, :k]

    for q in query_local_ids:
        if q not in gt_local_map:
            continue

        gt_set = set(gt_local_map[q][:k])
        pred_set = set(pred_neighbors[q][:k])

        hits.append(1.0 if len(gt_set & pred_set) > 0 else 0.0)

    return float(np.mean(hits)) if hits else 0.0

# -----------------------------
# 4) Compute metrics on VAL queries
# -----------------------------
r5_gnn  = recall_at_k_for_queries(sim, gt_local, val_query_locals, k=5)
r10_gnn = recall_at_k_for_queries(sim, gt_local, val_query_locals, k=10)
h10_gnn = hitrate_at_k_for_queries(sim, gt_local, val_query_locals, k=10)

r5_cnt  = recall_at_k_for_queries(sim_count, gt_local, val_query_locals, k=5)
r10_cnt = recall_at_k_for_queries(sim_count, gt_local, val_query_locals, k=10)
h10_cnt = hitrate_at_k_for_queries(sim_count, gt_local, val_query_locals, k=10)

print("\n=== REAL GT RETRIEVAL EVALUATION (VAL QUERIES) ===")
print(f"Node-count baseline Recall@5  : {r5_cnt:.4f}")
print(f"Node-count baseline Recall@10 : {r10_cnt:.4f}")
print(f"Node-count baseline Hit@10    : {h10_cnt:.4f}")
print(f"GNN embedding Recall@5        : {r5_gnn:.4f}")
print(f"GNN embedding Recall@10       : {r10_gnn:.4f}")
print(f"GNN embedding Hit@10          : {h10_gnn:.4f}")

=== FOCUS EMBEDDING SUMMARY ===
focus_embeddings shape: (5775, 128)
num focused graphs    : 5775

=== REAL GT RETRIEVAL EVALUATION (VAL QUERIES) ===
Node-count baseline Recall@5  : 0.0000
Node-count baseline Recall@10 : 0.0000
Node-count baseline Hit@10    : 0.0000
GNN embedding Recall@5        : 0.0069
GNN embedding Recall@10       : 0.0034
GNN embedding Hit@10          : 0.0345


In [29]:
# ============================================================
# CELL 16: Build real GT pair dataset for Jaccard scoring
# ============================================================

import random
import numpy as np
import torch
from torch.utils.data import Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ------------------------------------------------------------
# Pair construction
#   positive pairs: (query, true GT neighbor)
#   negative pairs: (query, sampled non-neighbor)
# label:
#   1.0 for positive
#   0.0 for negative
# ------------------------------------------------------------
def build_pair_lists(gt_local, query_local_ids, total_graphs, negatives_per_query=10):
    pos_pairs = []
    neg_pairs = []

    for q in query_local_ids:
        pos_set = set(gt_local[q])
        forbidden = set(pos_set)
        forbidden.add(q)

        # positives
        for p in pos_set:
            pos_pairs.append((q, p, 1.0))

        # negatives
        neg_candidates = [i for i in range(total_graphs) if i not in forbidden]
        if len(neg_candidates) == 0:
            continue

        sample_n = min(negatives_per_query * max(1, len(pos_set)), len(neg_candidates))
        sampled_negs = random.sample(neg_candidates, sample_n)

        for n in sampled_negs:
            neg_pairs.append((q, n, 0.0))

    return pos_pairs, neg_pairs


train_pos_pairs, train_neg_pairs = build_pair_lists(
    gt_local=gt_local,
    query_local_ids=train_query_locals,
    total_graphs=len(pyg_graphs_focus),
    negatives_per_query=3
)

val_pos_pairs, val_neg_pairs = build_pair_lists(
    gt_local=gt_local,
    query_local_ids=val_query_locals,
    total_graphs=len(pyg_graphs_focus),
    negatives_per_query=3
)

print("=== PAIR COUNTS ===")
print("Train positives:", len(train_pos_pairs))
print("Train negatives:", len(train_neg_pairs))
print("Val positives  :", len(val_pos_pairs))
print("Val negatives  :", len(val_neg_pairs))


class RealGTPairDataset(Dataset):
    def __init__(self, graphs, pair_tuples):
        self.graphs = graphs
        self.pairs = pair_tuples[:]  # [(a, b, label), ...]

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        a, b, y = self.pairs[idx]
        ga = self.graphs[a]
        gb = self.graphs[b]
        y = torch.tensor([y], dtype=torch.float32)
        return ga, gb, y


train_pair_tuples = train_pos_pairs + train_neg_pairs
val_pair_tuples   = val_pos_pairs + val_neg_pairs

random.shuffle(train_pair_tuples)
random.shuffle(val_pair_tuples)

train_pair_ds = RealGTPairDataset(pyg_graphs_focus, train_pair_tuples)
val_pair_ds   = RealGTPairDataset(pyg_graphs_focus, val_pair_tuples)

print("\n=== DATASET CHECK ===")
ga, gb, y = train_pair_ds[0]
print("A global:", ga.global_idx)
print("B global:", gb.global_idx)
print("Label   :", float(y.item()))
print("A shape :", tuple(ga.x.shape))
print("B shape :", tuple(gb.x.shape))

=== PAIR COUNTS ===
Train positives: 4449
Train negatives: 13347
Val positives  : 1136
Val negatives  : 3408

=== DATASET CHECK ===
A global: 187104
B global: 143784
Label   : 1.0
A shape : (100, 6)
B shape : (4, 6)


In [30]:
# ============================================================
# CELL 17: Polygon encoder + pairwise Jaccard head
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.loader import DataLoader as PyGDataLoader

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)


class PolygonEncoder(nn.Module):
    def __init__(self, in_dim=6, hidden_dim=64, emb_dim=128, dropout=0.10):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.conv3 = GCNConv(hidden_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.proj = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, emb_dim),
        )

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        x = F.relu(self.conv1(x, edge_index))
        x = self.dropout(x)

        x = F.relu(self.conv2(x, edge_index))
        x = self.dropout(x)

        x = F.relu(self.conv3(x, edge_index))

        g = global_mean_pool(x, batch)
        z = self.proj(g)
        return z


class JaccardHead(nn.Module):
    def __init__(self, emb_dim=128, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(emb_dim * 4, hidden),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden // 2, 1),
            nn.Sigmoid(),
        )

    def forward(self, h_a, h_b):
        x = torch.cat([h_a, h_b, torch.abs(h_a - h_b), h_a * h_b], dim=-1)
        return self.net(x)


class PolygonGNNJaccard(nn.Module):
    def __init__(self, in_dim=6, hidden_dim=64, emb_dim=128, ann_hidden=256, dropout=0.10):
        super().__init__()
        self.encoder = PolygonEncoder(in_dim, hidden_dim, emb_dim, dropout)
        self.head = JaccardHead(emb_dim, ann_hidden)

    def encode(self, batch):
        return self.encoder(batch)

    def forward(self, batch_a, batch_b):
        h_a = self.encode(batch_a)
        h_b = self.encode(batch_b)
        return self.head(h_a, h_b)


pair_model = PolygonGNNJaccard(
    in_dim=6,
    hidden_dim=64,
    emb_dim=128,
    ann_hidden=256,
    dropout=0.10
).to(DEVICE)

print(pair_model)

Using device: cuda
PolygonGNNJaccard(
  (encoder): PolygonEncoder(
    (conv1): GCNConv(6, 64)
    (conv2): GCNConv(64, 64)
    (conv3): GCNConv(64, 64)
    (dropout): Dropout(p=0.1, inplace=False)
    (proj): Sequential(
      (0): Linear(in_features=64, out_features=64, bias=True)
      (1): ReLU()
      (2): Dropout(p=0.1, inplace=False)
      (3): Linear(in_features=64, out_features=128, bias=True)
    )
  )
  (head): JaccardHead(
    (net): Sequential(
      (0): Linear(in_features=512, out_features=256, bias=True)
      (1): ReLU()
      (2): Dropout(p=0.2, inplace=False)
      (3): Linear(in_features=256, out_features=128, bias=True)
      (4): ReLU()
      (5): Dropout(p=0.1, inplace=False)
      (6): Linear(in_features=128, out_features=1, bias=True)
      (7): Sigmoid()
    )
  )
)


Epoch 01 | train_loss=0.567068 | val_loss=0.566123
Epoch 02 | train_loss=0.564862 | val_loss=0.562410
Epoch 03 | train_loss=0.564610 | val_loss=0.562463
Epoch 04 | train_loss=0.565030 | val_loss=0.562379
Epoch 05 | train_loss=0.564383 | val_loss=0.562799
Epoch 06 | train_loss=0.564035 | val_loss=0.562969
Epoch 07 | train_loss=0.564678 | val_loss=0.563213
Epoch 08 | train_loss=0.563901 | val_loss=0.562749
Epoch 09 | train_loss=0.564090 | val_loss=0.562465
Epoch 10 | train_loss=0.564226 | val_loss=0.562353


In [33]:
# ============================================================
# CELL 16B: Build real GT pair dataset with exact IoU labels
# ============================================================

import random
import numpy as np
import torch
from torch.utils.data import Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

def exact_jaccard_iou(geom_a, geom_b):
    try:
        inter = geom_a.intersection(geom_b).area
        union = geom_a.union(geom_b).area
        return float(inter / union) if union > 1e-12 else 0.0
    except Exception:
        return 0.0

def build_exact_iou_pairs(
    gt_local, query_local_ids, geoms_focus,
    negatives_per_positive=2,
    max_pos_per_query=10
):
    pair_tuples = []

    total_graphs = len(geoms_focus)

    for q in query_local_ids:
        pos_list = gt_local[q][:max_pos_per_query]
        pos_set = set(pos_list)
        forbidden = set(pos_set)
        forbidden.add(q)

        # positives with exact IoU
        for p in pos_list:
            y = exact_jaccard_iou(geoms_focus[q], geoms_focus[p])
            pair_tuples.append((q, p, y))

        # sampled negatives with exact IoU too
        neg_candidates = [i for i in range(total_graphs) if i not in forbidden]
        sample_n = min(len(neg_candidates), negatives_per_positive * max(1, len(pos_list)))
        sampled_negs = random.sample(neg_candidates, sample_n)

        for n in sampled_negs:
            y = exact_jaccard_iou(geoms_focus[q], geoms_focus[n])  # often 0, but exact
            pair_tuples.append((q, n, y))

    return pair_tuples


train_pair_tuples = build_exact_iou_pairs(
    gt_local=gt_local,
    query_local_ids=train_query_locals,
    geoms_focus=geoms_focus,
    negatives_per_positive=2,
    max_pos_per_query=10
)

val_pair_tuples = build_exact_iou_pairs(
    gt_local=gt_local,
    query_local_ids=val_query_locals,
    geoms_focus=geoms_focus,
    negatives_per_positive=2,
    max_pos_per_query=10
)

print("=== EXACT IOU PAIR COUNTS ===")
print("Train pairs:", len(train_pair_tuples))
print("Val pairs  :", len(val_pair_tuples))

train_labels = np.array([y for _, _, y in train_pair_tuples], dtype=np.float32)
val_labels   = np.array([y for _, _, y in val_pair_tuples], dtype=np.float32)

print("\nTrain label stats:")
print("  min   :", float(train_labels.min()))
print("  p50   :", float(np.median(train_labels)))
print("  p90   :", float(np.percentile(train_labels, 90)))
print("  max   :", float(train_labels.max()))

print("\nVal label stats:")
print("  min   :", float(val_labels.min()))
print("  p50   :", float(np.median(val_labels)))
print("  p90   :", float(np.percentile(val_labels, 90)))
print("  max   :", float(val_labels.max()))


class RealGTPairDatasetIoU(Dataset):
    def __init__(self, graphs, pair_tuples):
        self.graphs = graphs
        self.pairs = pair_tuples

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        a, b, y = self.pairs[idx]
        ga = self.graphs[a]
        gb = self.graphs[b]
        y = torch.tensor([y], dtype=torch.float32)
        return ga, gb, y


train_pair_ds = RealGTPairDatasetIoU(pyg_graphs_focus, train_pair_tuples)
val_pair_ds   = RealGTPairDatasetIoU(pyg_graphs_focus, val_pair_tuples)

print("\n=== DATASET CHECK ===")
ga, gb, y = train_pair_ds[0]
print("A global:", ga.global_idx)
print("B global:", gb.global_idx)
print("IoU     :", float(y.item()))

=== EXACT IOU PAIR COUNTS ===
Train pairs: 6750
Val pairs  : 1734

Train label stats:
  min   : 0.0
  p50   : 0.0
  p90   : 0.0
  max   : 0.0

Val label stats:
  min   : 0.0
  p50   : 0.0
  p90   : 0.0
  max   : 0.0

=== DATASET CHECK ===
A global: 187309
B global: 147265
IoU     : 0.0


In [34]:
# ============================================================
# CELL 16C: Canonical shape IoU instead of raw geographic IoU
# ============================================================

import random
import numpy as np
import torch
from torch.utils.data import Dataset
from shapely import affinity
from shapely.geometry import Polygon, MultiPolygon

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

def largest_polygon_part(geom):
    if geom.geom_type == "Polygon":
        return geom
    if geom.geom_type == "MultiPolygon":
        parts = list(geom.geoms)
        if not parts:
            return None
        return max(parts, key=lambda p: p.area)
    return None

def canonicalize_polygon(geom):
    """
    Make polygon location/scale invariant.
    Keeps orientation as-is for now.
    """
    poly = largest_polygon_part(geom)
    if poly is None or poly.is_empty:
        return None

    # fix invalid
    if not poly.is_valid:
        poly = poly.buffer(0)
        if poly.is_empty:
            return None
        poly = largest_polygon_part(poly)
        if poly is None:
            return None

    c = poly.centroid
    poly = affinity.translate(poly, xoff=-c.x, yoff=-c.y)

    minx, miny, maxx, maxy = poly.bounds
    w = maxx - minx
    h = maxy - miny
    scale = max(w, h)

    if scale < 1e-12:
        return None

    poly = affinity.scale(poly, xfact=1.0 / scale, yfact=1.0 / scale, origin=(0, 0))
    return poly

def shape_iou_with_rotations(geom_a, geom_b, angles=(0, 45, 90, 135)):
    """
    Compute IoU after canonical translation/scale normalization,
    then try a few rotations and keep the best IoU.
    """
    pa = canonicalize_polygon(geom_a)
    pb = canonicalize_polygon(geom_b)

    if pa is None or pb is None:
        return 0.0

    best = 0.0
    for ang in angles:
        pb_rot = affinity.rotate(pb, ang, origin=(0, 0))
        try:
            inter = pa.intersection(pb_rot).area
            union = pa.union(pb_rot).area
            iou = float(inter / union) if union > 1e-12 else 0.0
            if iou > best:
                best = iou
        except Exception:
            continue

    return best

def build_shape_iou_pairs(
    gt_local, query_local_ids, geoms_focus,
    negatives_per_positive=2,
    max_pos_per_query=10
):
    pair_tuples = []

    total_graphs = len(geoms_focus)

    for q in query_local_ids:
        pos_list = gt_local[q][:max_pos_per_query]
        pos_set = set(pos_list)
        forbidden = set(pos_set)
        forbidden.add(q)

        # positives
        for p in pos_list:
            y = shape_iou_with_rotations(geoms_focus[q], geoms_focus[p])
            pair_tuples.append((q, p, y))

        # negatives
        neg_candidates = [i for i in range(total_graphs) if i not in forbidden]
        sample_n = min(len(neg_candidates), negatives_per_positive * max(1, len(pos_list)))
        sampled_negs = random.sample(neg_candidates, sample_n)

        for n in sampled_negs:
            y = shape_iou_with_rotations(geoms_focus[q], geoms_focus[n])
            pair_tuples.append((q, n, y))

    return pair_tuples


train_pair_tuples = build_shape_iou_pairs(
    gt_local=gt_local,
    query_local_ids=train_query_locals,
    geoms_focus=geoms_focus,
    negatives_per_positive=2,
    max_pos_per_query=10
)

val_pair_tuples = build_shape_iou_pairs(
    gt_local=gt_local,
    query_local_ids=val_query_locals,
    geoms_focus=geoms_focus,
    negatives_per_positive=2,
    max_pos_per_query=10
)

print("=== SHAPE IOU PAIR COUNTS ===")
print("Train pairs:", len(train_pair_tuples))
print("Val pairs  :", len(val_pair_tuples))

train_labels = np.array([y for _, _, y in train_pair_tuples], dtype=np.float32)
val_labels   = np.array([y for _, _, y in val_pair_tuples], dtype=np.float32)

print("\nTrain label stats:")
print("  min   :", float(train_labels.min()))
print("  p50   :", float(np.median(train_labels)))
print("  p90   :", float(np.percentile(train_labels, 90)))
print("  p99   :", float(np.percentile(train_labels, 99)))
print("  max   :", float(train_labels.max()))
print("  nonzero:", int((train_labels > 0).sum()), "/", len(train_labels))

print("\nVal label stats:")
print("  min   :", float(val_labels.min()))
print("  p50   :", float(np.median(val_labels)))
print("  p90   :", float(np.percentile(val_labels, 90)))
print("  p99   :", float(np.percentile(val_labels, 99)))
print("  max   :", float(val_labels.max()))
print("  nonzero:", int((val_labels > 0).sum()), "/", len(val_labels))


class RealGTPairDatasetIoU(Dataset):
    def __init__(self, graphs, pair_tuples):
        self.graphs = graphs
        self.pairs = pair_tuples

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        a, b, y = self.pairs[idx]
        ga = self.graphs[a]
        gb = self.graphs[b]
        y = torch.tensor([y], dtype=torch.float32)
        return ga, gb, y


train_pair_ds = RealGTPairDatasetIoU(pyg_graphs_focus, train_pair_tuples)
val_pair_ds   = RealGTPairDatasetIoU(pyg_graphs_focus, val_pair_tuples)

print("\n=== DATASET CHECK ===")
ga, gb, y = train_pair_ds[0]
print("A global:", ga.global_idx)
print("B global:", gb.global_idx)
print("Shape IoU:", float(y.item()))

=== SHAPE IOU PAIR COUNTS ===
Train pairs: 6750
Val pairs  : 1734

Train label stats:
  min   : 0.005272307433187962
  p50   : 0.4761723279953003
  p90   : 0.6913719773292542
  p99   : 0.8094836473464966
  max   : 0.9645805358886719
  nonzero: 6750 / 6750

Val label stats:
  min   : 0.03140264004468918
  p50   : 0.488321989774704
  p90   : 0.68106609582901
  p99   : 0.7873934507369995
  max   : 0.9243460297584534
  nonzero: 1734 / 1734

=== DATASET CHECK ===
A global: 187309
B global: 147265
Shape IoU: 0.7977422475814819


In [35]:
# ============================================================
# CELL 18B: Train pairwise IoU regressor
# ============================================================

import numpy as np
import random
import torch
import torch.nn as nn

def make_pair_batches(dataset, batch_size=16, shuffle=True):
    indices = list(range(len(dataset)))
    if shuffle:
        random.shuffle(indices)

    for start in range(0, len(indices), batch_size):
        batch_idx = indices[start:start + batch_size]

        list_a, list_b, ys = [], [], []
        for idx in batch_idx:
            ga, gb, y = dataset[idx]
            list_a.append(ga)
            list_b.append(gb)
            ys.append(y)

        batch_a = next(iter(PyGDataLoader(list_a, batch_size=len(list_a), shuffle=False)))
        batch_b = next(iter(PyGDataLoader(list_b, batch_size=len(list_b), shuffle=False)))
        y = torch.cat(ys, dim=0).view(-1, 1)   # [B, 1]

        yield batch_a, batch_b, y


criterion = nn.MSELoss()
optimizer = torch.optim.Adam(pair_model.parameters(), lr=1e-3, weight_decay=1e-5)

EPOCHS = 12
BATCH_SIZE = 16

train_history = []
val_history = []

for epoch in range(1, EPOCHS + 1):
    pair_model.train()
    train_losses = []

    for batch_a, batch_b, y in make_pair_batches(train_pair_ds, batch_size=BATCH_SIZE, shuffle=True):
        batch_a = batch_a.to(DEVICE)
        batch_b = batch_b.to(DEVICE)
        y = y.to(DEVICE)

        pred = pair_model(batch_a, batch_b)
        pred = torch.clamp(pred, 0.0, 1.0)

        loss = criterion(pred, y)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(pair_model.parameters(), 1.0)
        optimizer.step()

        train_losses.append(loss.item())

    train_mse = float(np.mean(train_losses)) if train_losses else float("nan")
    train_history.append(train_mse)

    pair_model.eval()
    val_losses = []

    with torch.no_grad():
        for batch_a, batch_b, y in make_pair_batches(val_pair_ds, batch_size=BATCH_SIZE, shuffle=False):
            batch_a = batch_a.to(DEVICE)
            batch_b = batch_b.to(DEVICE)
            y = y.to(DEVICE)

            pred = pair_model(batch_a, batch_b)
            pred = torch.clamp(pred, 0.0, 1.0)

            loss = criterion(pred, y)
            val_losses.append(loss.item())

    val_mse = float(np.mean(val_losses)) if val_losses else float("nan")
    val_history.append(val_mse)

    print(f"Epoch {epoch:02d} | train_mse={train_mse:.6f} | val_mse={val_mse:.6f}")

Epoch 01 | train_mse=0.035315 | val_mse=0.031040
Epoch 02 | train_mse=0.034564 | val_mse=0.030920
Epoch 03 | train_mse=0.034515 | val_mse=0.030997
Epoch 04 | train_mse=0.034538 | val_mse=0.031043
Epoch 05 | train_mse=0.034558 | val_mse=0.030966
Epoch 06 | train_mse=0.034526 | val_mse=0.030965
Epoch 07 | train_mse=0.034534 | val_mse=0.030890
Epoch 08 | train_mse=0.034538 | val_mse=0.030905
Epoch 09 | train_mse=0.034532 | val_mse=0.030916
Epoch 10 | train_mse=0.034533 | val_mse=0.030890
Epoch 11 | train_mse=0.034526 | val_mse=0.030899
Epoch 12 | train_mse=0.034523 | val_mse=0.030857


In [36]:
# ============================================================
# CELL 19: Real GT retrieval using predicted shape-IoU scores
# ============================================================

import numpy as np
import torch
from torch_geometric.loader import DataLoader as PyGDataLoader

pair_model.eval()

# ------------------------------------------------------------
# 1) Precompute embeddings for all focused graphs
# ------------------------------------------------------------
focus_loader = PyGDataLoader(pyg_graphs_focus, batch_size=128, shuffle=False)

all_h = []
with torch.no_grad():
    for batch in focus_loader:
        batch = batch.to(DEVICE)
        h = pair_model.encode(batch)
        all_h.append(h.cpu())

H = torch.cat(all_h, dim=0)   # [N, D]
print("Embedding bank:", tuple(H.shape))


# ------------------------------------------------------------
# 2) Score one query against all graphs using the learned head
# ------------------------------------------------------------
def score_pairs_from_embeddings(H_query, H_all, chunk_size=1024):
    scores = []
    with torch.no_grad():
        for start in range(0, H_all.shape[0], chunk_size):
            end = min(start + chunk_size, H_all.shape[0])
            h_b = H_all[start:end].to(DEVICE)
            h_a = H_query.unsqueeze(0).expand(h_b.shape[0], -1).to(DEVICE)

            s = pair_model.head(h_a, h_b)
            s = torch.clamp(s, 0.0, 1.0).squeeze(-1).cpu().numpy()
            scores.append(s)

    return np.concatenate(scores, axis=0)


# ------------------------------------------------------------
# 3) Ranking metrics
# ------------------------------------------------------------
def recall_at_k_ranked(pred_rankings, gt_local_map, query_local_ids, k=10):
    vals = []
    for q in query_local_ids:
        gt_set = set(gt_local_map[q][:k])
        pred_set = set(pred_rankings[q][:k])
        vals.append(len(gt_set & pred_set) / max(len(gt_set), 1))
    return float(np.mean(vals))

def hit_at_k_ranked(pred_rankings, gt_local_map, query_local_ids, k=10):
    vals = []
    for q in query_local_ids:
        gt_set = set(gt_local_map[q][:k])
        pred_set = set(pred_rankings[q][:k])
        vals.append(1.0 if len(gt_set & pred_set) > 0 else 0.0)
    return float(np.mean(vals))


# ------------------------------------------------------------
# 4) Predict rankings for validation queries
# ------------------------------------------------------------
pred_rankings = {}

for q in val_query_locals:
    scores = score_pairs_from_embeddings(H[q], H, chunk_size=1024)
    scores[q] = -1e9
    ranked = np.argsort(-scores)[:10]
    pred_rankings[q] = ranked.tolist()


# ------------------------------------------------------------
# 5) Baseline: node-count
# ------------------------------------------------------------
focus_num_nodes = np.asarray([int(g.num_nodes_manual) for g in pyg_graphs_focus], dtype=np.int32)

baseline_rankings = {}
for q in val_query_locals:
    s = -np.abs(focus_num_nodes - focus_num_nodes[q]).astype(np.float32)
    s[q] = -1e9
    baseline_rankings[q] = np.argsort(-s)[:10].tolist()


# ------------------------------------------------------------
# 6) Metrics
# ------------------------------------------------------------
r10_pair = recall_at_k_ranked(pred_rankings, gt_local, val_query_locals, k=10)
h10_pair = hit_at_k_ranked(pred_rankings, gt_local, val_query_locals, k=10)

r10_base = recall_at_k_ranked(baseline_rankings, gt_local, val_query_locals, k=10)
h10_base = hit_at_k_ranked(baseline_rankings, gt_local, val_query_locals, k=10)

print("\n=== REAL GT SHAPE-IOU RERANK EVAL ===")
print(f"Node-count baseline Recall@10 : {r10_base:.4f}")
print(f"Node-count baseline Hit@10    : {h10_base:.4f}")
print(f"Pairwise Shape-IoU Recall@10  : {r10_pair:.4f}")
print(f"Pairwise Shape-IoU Hit@10     : {h10_pair:.4f}")


# ------------------------------------------------------------
# 7) Show a few query examples
# ------------------------------------------------------------
print("\n=== SAMPLE VAL QUERY EXAMPLES ===")
for q in val_query_locals[:3]:
    print(f"\nlocal_q={q} global_q={l2g[q]}")
    print("GT top10 globals   :", [l2g[x] for x in gt_local[q][:10]])
    print("Pred top10 globals :", [l2g[x] for x in pred_rankings[q][:10]])

Embedding bank: (5775, 128)

=== REAL GT SHAPE-IOU RERANK EVAL ===
Node-count baseline Recall@10 : 0.0000
Node-count baseline Hit@10    : 0.0000
Pairwise Shape-IoU Recall@10  : 0.0017
Pairwise Shape-IoU Hit@10     : 0.0172

=== SAMPLE VAL QUERY EXAMPLES ===

local_q=5738 global_q=187282
GT top10 globals   : [183778, 62218, 183799, 156861, 144868, 71191, 97469, 182612, 180700, 134169]
Pred top10 globals : [131717, 132099, 132046, 131991, 131990, 131989, 131964, 131960, 131948, 131818]

local_q=5624 global_q=187168
GT top10 globals   : [176959, 22051, 98011, 144662, 149799, 123303, 90397, 132996, 122587, 78485]
Pred top10 globals : [131717, 132099, 132046, 131991, 131990, 131989, 131964, 131960, 131948, 131818]

local_q=5567 global_q=187111
GT top10 globals   : [103601, 166163, 48071, 3843, 35951, 28125, 62149, 112588, 122421, 54898]
Pred top10 globals : [131717, 132099, 132046, 131991, 131990, 131989, 131964, 131960, 131948, 131818]


In [37]:
# ============================================================
# CELL 16D: Build GT-ranking triplets for direct ranking loss
# ============================================================

import random
from torch.utils.data import Dataset

SEED = 42
random.seed(SEED)

class RealGTRankingDataset(Dataset):
    def __init__(self, graphs, gt_local, query_local_ids, top_pos_k=10, negs_per_query=5):
        self.graphs = graphs
        self.samples = []

        total_graphs = len(graphs)

        for q in query_local_ids:
            pos_list = gt_local[q][:top_pos_k]
            pos_set = set(pos_list)
            forbidden = set(pos_set)
            forbidden.add(q)

            neg_candidates = [i for i in range(total_graphs) if i not in forbidden]
            if not neg_candidates:
                continue

            # sample a few negatives per positive
            sampled_negs = random.sample(
                neg_candidates,
                min(len(neg_candidates), negs_per_query * max(1, len(pos_list)))
            )

            for p in pos_list:
                for n in sampled_negs[:negs_per_query]:
                    self.samples.append((q, p, n))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        q, p, n = self.samples[idx]
        return self.graphs[q], self.graphs[p], self.graphs[n]


train_rank_ds = RealGTRankingDataset(
    graphs=pyg_graphs_focus,
    gt_local=gt_local,
    query_local_ids=train_query_locals,
    top_pos_k=10,
    negs_per_query=3
)

val_rank_ds = RealGTRankingDataset(
    graphs=pyg_graphs_focus,
    gt_local=gt_local,
    query_local_ids=val_query_locals,
    top_pos_k=10,
    negs_per_query=3
)

print("=== RANKING DATASET ===")
print("Train ranking triples:", len(train_rank_ds))
print("Val ranking triples  :", len(val_rank_ds))

q, p, n = train_rank_ds[0]
print("\nSample triple:")
print("Q global:", q.global_idx)
print("P global:", p.global_idx)
print("N global:", n.global_idx)

=== RANKING DATASET ===
Train ranking triples: 6750
Val ranking triples  : 1734

Sample triple:
Q global: 187309
P global: 147265
N global: 179109


In [38]:
# ============================================================
# CELL 18: Train pairwise Jaccard scorer
# ============================================================

import numpy as np
import random
import torch
import torch.nn as nn

def make_pair_batches(dataset, batch_size=48, shuffle=True):
    indices = list(range(len(dataset)))
    if shuffle:
        random.shuffle(indices)

    for start in range(0, len(indices), batch_size):
        batch_idx = indices[start:start + batch_size]

        list_a, list_b, ys = [], [], []
        for idx in batch_idx:
            ga, gb, y = dataset[idx]
            list_a.append(ga)
            list_b.append(gb)
            ys.append(y)

        batch_a = next(iter(PyGDataLoader(list_a, batch_size=len(list_a), shuffle=False)))
        batch_b = next(iter(PyGDataLoader(list_b, batch_size=len(list_b), shuffle=False)))
        y = torch.cat(ys, dim=0).view(-1, 1)  # [B, 1]

        yield batch_a, batch_b, y


criterion = nn.BCELoss()
optimizer = torch.optim.Adam(pair_model.parameters(), lr=1e-3, weight_decay=1e-5)

EPOCHS = 10
BATCH_SIZE = 16

for epoch in range(1, EPOCHS + 1):
    pair_model.train()
    train_losses = []

    for batch_a, batch_b, y in make_pair_batches(train_pair_ds, batch_size=BATCH_SIZE, shuffle=True):
        batch_a = batch_a.to(DEVICE)
        batch_b = batch_b.to(DEVICE)
        y = y.to(DEVICE)

        pred = pair_model(batch_a, batch_b)
        loss = criterion(pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())

    pair_model.eval()
    val_losses = []

    with torch.no_grad():
        for batch_a, batch_b, y in make_pair_batches(val_pair_ds, batch_size=BATCH_SIZE, shuffle=False):
            batch_a = batch_a.to(DEVICE)
            batch_b = batch_b.to(DEVICE)
            y = y.to(DEVICE)

            pred = pair_model(batch_a, batch_b)
            loss = criterion(pred, y)
            val_losses.append(loss.item())

    print(f"Epoch {epoch:02d} | train_loss={np.mean(train_losses):.6f} | val_loss={np.mean(val_losses):.6f}")

Epoch 01 | train_loss=0.690037 | val_loss=0.691160
Epoch 02 | train_loss=0.690047 | val_loss=0.691058
Epoch 03 | train_loss=0.690052 | val_loss=0.691116
Epoch 04 | train_loss=0.690063 | val_loss=0.691099
Epoch 05 | train_loss=0.690032 | val_loss=0.691010
Epoch 06 | train_loss=0.690092 | val_loss=0.691079
Epoch 07 | train_loss=0.690050 | val_loss=0.691147
Epoch 08 | train_loss=0.690031 | val_loss=0.691071
Epoch 09 | train_loss=0.690063 | val_loss=0.691099
Epoch 10 | train_loss=0.690049 | val_loss=0.691124


In [39]:
# ============================================================
# CELL 18D: Train scorer with pairwise ranking loss
# score(q, pos) should be > score(q, neg)
# ============================================================

import numpy as np
import random
import torch
import torch.nn as nn

def make_rank_batches(dataset, batch_size=16, shuffle=True):
    indices = list(range(len(dataset)))
    if shuffle:
        random.shuffle(indices)

    for start in range(0, len(indices), batch_size):
        batch_idx = indices[start:start + batch_size]

        list_q, list_p, list_n = [], [], []
        for idx in batch_idx:
            gq, gp, gn = dataset[idx]
            list_q.append(gq)
            list_p.append(gp)
            list_n.append(gn)

        batch_q = next(iter(PyGDataLoader(list_q, batch_size=len(list_q), shuffle=False)))
        batch_p = next(iter(PyGDataLoader(list_p, batch_size=len(list_p), shuffle=False)))
        batch_n = next(iter(PyGDataLoader(list_n, batch_size=len(list_n), shuffle=False)))

        yield batch_q, batch_p, batch_n


optimizer = torch.optim.Adam(pair_model.parameters(), lr=1e-3, weight_decay=1e-5)

EPOCHS = 12
BATCH_SIZE = 16
MARGIN = 0.2

for epoch in range(1, EPOCHS + 1):
    pair_model.train()
    train_losses = []

    for batch_q, batch_p, batch_n in make_rank_batches(train_rank_ds, batch_size=BATCH_SIZE, shuffle=True):
        batch_q = batch_q.to(DEVICE)
        batch_p = batch_p.to(DEVICE)
        batch_n = batch_n.to(DEVICE)

        s_pos = pair_model(batch_q, batch_p)   # [B,1]
        s_neg = pair_model(batch_q, batch_n)   # [B,1]

        # hinge ranking loss
        loss = torch.relu(MARGIN - s_pos + s_neg).mean()

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(pair_model.parameters(), 1.0)
        optimizer.step()

        train_losses.append(loss.item())

    pair_model.eval()
    val_losses = []

    with torch.no_grad():
        for batch_q, batch_p, batch_n in make_rank_batches(val_rank_ds, batch_size=BATCH_SIZE, shuffle=False):
            batch_q = batch_q.to(DEVICE)
            batch_p = batch_p.to(DEVICE)
            batch_n = batch_n.to(DEVICE)

            s_pos = pair_model(batch_q, batch_p)
            s_neg = pair_model(batch_q, batch_n)

            loss = torch.relu(MARGIN - s_pos + s_neg).mean()
            val_losses.append(loss.item())

    print(f"Epoch {epoch:02d} | train_rankloss={np.mean(train_losses):.6f} | val_rankloss={np.mean(val_losses):.6f}")

Epoch 01 | train_rankloss=0.199941 | val_rankloss=0.200000
Epoch 02 | train_rankloss=0.198749 | val_rankloss=0.200000
Epoch 03 | train_rankloss=0.200281 | val_rankloss=0.200000
Epoch 04 | train_rankloss=0.198750 | val_rankloss=0.200000


KeyboardInterrupt: 

In [41]:
# ============================================================
# CELL 17R: Fresh ranking model (NO sigmoid, NO batchnorm)
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)


class PolygonEncoder(nn.Module):
    def __init__(self, in_dim=6, hidden_dim=64, emb_dim=128, dropout=0.10):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.conv3 = GCNConv(hidden_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout)

        self.proj = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, emb_dim),
        )

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        x = F.relu(self.conv1(x, edge_index))
        x = self.dropout(x)

        x = F.relu(self.conv2(x, edge_index))
        x = self.dropout(x)

        x = F.relu(self.conv3(x, edge_index))

        g = global_mean_pool(x, batch)
        z = self.proj(g)
        return z


class RankingHead(nn.Module):
    def __init__(self, emb_dim=128, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(emb_dim * 4, hidden),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden // 2, 1)
        )

    def forward(self, h_a, h_b):
        x = torch.cat([h_a, h_b, torch.abs(h_a - h_b), h_a * h_b], dim=-1)
        return self.net(x)   # raw score


class PolygonRankingModel(nn.Module):
    def __init__(self, in_dim=6, hidden_dim=64, emb_dim=128, head_hidden=256, dropout=0.10):
        super().__init__()
        self.encoder = PolygonEncoder(in_dim, hidden_dim, emb_dim, dropout)
        self.head = RankingHead(emb_dim, head_hidden)

    def encode(self, batch):
        return self.encoder(batch)

    def forward(self, batch_a, batch_b):
        h_a = self.encode(batch_a)
        h_b = self.encode(batch_b)
        return self.head(h_a, h_b)


# IMPORTANT: fresh model
pair_model = PolygonRankingModel(
    in_dim=6,
    hidden_dim=64,
    emb_dim=128,
    head_hidden=256,
    dropout=0.10
).to(DEVICE)

print(pair_model)

Using device: cuda
PolygonRankingModel(
  (encoder): PolygonEncoder(
    (conv1): GCNConv(6, 64)
    (conv2): GCNConv(64, 64)
    (conv3): GCNConv(64, 64)
    (dropout): Dropout(p=0.1, inplace=False)
    (proj): Sequential(
      (0): Linear(in_features=64, out_features=64, bias=True)
      (1): ReLU()
      (2): Dropout(p=0.1, inplace=False)
      (3): Linear(in_features=64, out_features=128, bias=True)
    )
  )
  (head): RankingHead(
    (net): Sequential(
      (0): Linear(in_features=512, out_features=256, bias=True)
      (1): ReLU()
      (2): Dropout(p=0.2, inplace=False)
      (3): Linear(in_features=256, out_features=128, bias=True)
      (4): ReLU()
      (5): Dropout(p=0.1, inplace=False)
      (6): Linear(in_features=128, out_features=1, bias=True)
    )
  )
)


In [42]:
# ============================================================
# CELL 18R: Train fresh model with ranking loss
# ============================================================

import numpy as np
import random
import torch
from torch_geometric.loader import DataLoader as PyGDataLoader

def make_rank_batches(dataset, batch_size=16, shuffle=True):
    indices = list(range(len(dataset)))
    if shuffle:
        random.shuffle(indices)

    for start in range(0, len(indices), batch_size):
        batch_idx = indices[start:start + batch_size]

        list_q, list_p, list_n = [], [], []
        for idx in batch_idx:
            gq, gp, gn = dataset[idx]
            list_q.append(gq)
            list_p.append(gp)
            list_n.append(gn)

        batch_q = next(iter(PyGDataLoader(list_q, batch_size=len(list_q), shuffle=False)))
        batch_p = next(iter(PyGDataLoader(list_p, batch_size=len(list_p), shuffle=False)))
        batch_n = next(iter(PyGDataLoader(list_n, batch_size=len(list_n), shuffle=False)))

        yield batch_q, batch_p, batch_n


optimizer = torch.optim.Adam(pair_model.parameters(), lr=5e-4, weight_decay=1e-5)

EPOCHS = 12
BATCH_SIZE = 16
MARGIN = 0.1

for epoch in range(1, EPOCHS + 1):
    pair_model.train()
    train_losses = []

    for batch_q, batch_p, batch_n in make_rank_batches(train_rank_ds, batch_size=BATCH_SIZE, shuffle=True):
        batch_q = batch_q.to(DEVICE)
        batch_p = batch_p.to(DEVICE)
        batch_n = batch_n.to(DEVICE)

        s_pos = pair_model(batch_q, batch_p)   # [B,1], raw
        s_neg = pair_model(batch_q, batch_n)   # [B,1], raw

        loss = torch.relu(MARGIN - s_pos + s_neg).mean()

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(pair_model.parameters(), 1.0)
        optimizer.step()

        train_losses.append(loss.item())

    pair_model.eval()
    val_losses = []

    with torch.no_grad():
        for batch_q, batch_p, batch_n in make_rank_batches(val_rank_ds, batch_size=BATCH_SIZE, shuffle=False):
            batch_q = batch_q.to(DEVICE)
            batch_p = batch_p.to(DEVICE)
            batch_n = batch_n.to(DEVICE)

            s_pos = pair_model(batch_q, batch_p)
            s_neg = pair_model(batch_q, batch_n)

            loss = torch.relu(MARGIN - s_pos + s_neg).mean()
            val_losses.append(loss.item())

    print(f"Epoch {epoch:02d} | train_rankloss={np.mean(train_losses):.6f} | val_rankloss={np.mean(val_losses):.6f}")

Epoch 01 | train_rankloss=0.098988 | val_rankloss=0.101598
Epoch 02 | train_rankloss=0.097658 | val_rankloss=0.102960
Epoch 03 | train_rankloss=0.095903 | val_rankloss=0.105782
Epoch 04 | train_rankloss=0.094167 | val_rankloss=0.106930
Epoch 05 | train_rankloss=0.092764 | val_rankloss=0.103775
Epoch 06 | train_rankloss=0.091432 | val_rankloss=0.107728
Epoch 07 | train_rankloss=0.089752 | val_rankloss=0.108662
Epoch 08 | train_rankloss=0.089161 | val_rankloss=0.108213
Epoch 09 | train_rankloss=0.088815 | val_rankloss=0.106600
Epoch 10 | train_rankloss=0.087247 | val_rankloss=0.105951
Epoch 11 | train_rankloss=0.085630 | val_rankloss=0.111844
Epoch 12 | train_rankloss=0.083305 | val_rankloss=0.105820


In [43]:
# ============================================================
# CELL 19R: Evaluate fresh ranking model
# ============================================================

import numpy as np
import torch
from torch_geometric.loader import DataLoader as PyGDataLoader

pair_model.eval()

focus_loader = PyGDataLoader(pyg_graphs_focus, batch_size=128, shuffle=False)

all_h = []
with torch.no_grad():
    for batch in focus_loader:
        batch = batch.to(DEVICE)
        h = pair_model.encode(batch)
        all_h.append(h.cpu())

H = torch.cat(all_h, dim=0)   # [N, D]
print("Embedding bank:", tuple(H.shape))


def score_pairs_from_embeddings(H_query, H_all, chunk_size=1024):
    scores = []
    with torch.no_grad():
        for start in range(0, H_all.shape[0], chunk_size):
            end = min(start + chunk_size, H_all.shape[0])
            h_b = H_all[start:end].to(DEVICE)
            h_a = H_query.unsqueeze(0).expand(h_b.shape[0], -1).to(DEVICE)
            s = pair_model.head(h_a, h_b).squeeze(-1).cpu().numpy()
            scores.append(s)
    return np.concatenate(scores, axis=0)


def recall_at_k_ranked(pred_rankings, gt_local_map, query_local_ids, k=10):
    vals = []
    for q in query_local_ids:
        gt_set = set(gt_local_map[q][:k])
        pred_set = set(pred_rankings[q][:k])
        vals.append(len(gt_set & pred_set) / max(len(gt_set), 1))
    return float(np.mean(vals))

def hit_at_k_ranked(pred_rankings, gt_local_map, query_local_ids, k=10):
    vals = []
    for q in query_local_ids:
        gt_set = set(gt_local_map[q][:k])
        pred_set = set(pred_rankings[q][:k])
        vals.append(1.0 if len(gt_set & pred_set) > 0 else 0.0)
    return float(np.mean(vals))


pred_rankings = {}

for q in val_query_locals:
    scores = score_pairs_from_embeddings(H[q], H, chunk_size=1024)
    scores[q] = -1e9
    pred_rankings[q] = np.argsort(-scores)[:10].tolist()

focus_num_nodes = np.asarray([int(g.num_nodes_manual) for g in pyg_graphs_focus], dtype=np.int32)
baseline_rankings = {}

for q in val_query_locals:
    s = -np.abs(focus_num_nodes - focus_num_nodes[q]).astype(np.float32)
    s[q] = -1e9
    baseline_rankings[q] = np.argsort(-s)[:10].tolist()

r10_model = recall_at_k_ranked(pred_rankings, gt_local, val_query_locals, k=10)
h10_model = hit_at_k_ranked(pred_rankings, gt_local, val_query_locals, k=10)

r10_base = recall_at_k_ranked(baseline_rankings, gt_local, val_query_locals, k=10)
h10_base = hit_at_k_ranked(baseline_rankings, gt_local, val_query_locals, k=10)

print("\n=== REAL GT RANKING EVAL ===")
print(f"Node-count baseline Recall@10 : {r10_base:.4f}")
print(f"Node-count baseline Hit@10    : {h10_base:.4f}")
print(f"Ranking model Recall@10       : {r10_model:.4f}")
print(f"Ranking model Hit@10          : {h10_model:.4f}")

Embedding bank: (5775, 128)

=== REAL GT RANKING EVAL ===
Node-count baseline Recall@10 : 0.0000
Node-count baseline Hit@10    : 0.0000
Ranking model Recall@10       : 0.0034
Ranking model Hit@10          : 0.0345


In [44]:
# ============================================================
# CELL 20: Paper-lite geometric descriptor retrieval baseline
# uses normalized local + two-hop geometric statistics
# ============================================================

import numpy as np

def ring_descriptor_paper_lite(ring):
    """
    Build a fixed-length descriptor from a polygon ring.
    Inspired by the paper's focus on local geometric relationships.
    """
    pts = normalize_ring(ring)   # [N,2], open ring
    n = len(pts)

    # edge lengths
    nxt = np.roll(pts, -1, axis=0)
    edges = nxt - pts
    edge_len = np.linalg.norm(edges, axis=1)

    # turning angles
    turn_cos = []
    turn_sin = []

    # two-hop chord lengths
    chord2 = []

    for i in range(n):
        p_prev = pts[(i - 1) % n]
        p = pts[i]
        p_next = pts[(i + 1) % n]

        v1 = p - p_prev
        v2 = p_next - p

        nv1 = np.linalg.norm(v1)
        nv2 = np.linalg.norm(v2)

        if nv1 < 1e-8 or nv2 < 1e-8:
            c = 1.0
            s = 0.0
        else:
            u1 = v1 / nv1
            u2 = v2 / nv2
            c = np.clip(np.dot(u1, u2), -1.0, 1.0)
            s = u1[0] * u2[1] - u1[1] * u2[0]

        turn_cos.append(c)
        turn_sin.append(s)

        p_2hop = pts[(i + 2) % n]
        chord2.append(np.linalg.norm(p_2hop - p))

    edge_len = np.asarray(edge_len, dtype=np.float32)
    turn_cos = np.asarray(turn_cos, dtype=np.float32)
    turn_sin = np.asarray(turn_sin, dtype=np.float32)
    chord2 = np.asarray(chord2, dtype=np.float32)

    # normalize edge/chord scales
    if edge_len.sum() > 1e-8:
        edge_len = edge_len / edge_len.sum()
    if chord2.sum() > 1e-8:
        chord2 = chord2 / chord2.sum()

    def hist_feat(arr, bins, lo, hi):
        h, _ = np.histogram(arr, bins=bins, range=(lo, hi), density=False)
        h = h.astype(np.float32)
        if h.sum() > 0:
            h /= h.sum()
        return h

    feat = np.concatenate([
        np.array([
            n,
            np.median(edge_len),
            np.max(edge_len),
            np.median(chord2),
            np.max(chord2),
            np.median(turn_cos),
            np.median(turn_sin),
        ], dtype=np.float32),

        hist_feat(edge_len, bins=16, lo=0.0, hi=max(1e-6, edge_len.max() + 1e-6)),
        hist_feat(chord2,   bins=16, lo=0.0, hi=max(1e-6, chord2.max() + 1e-6)),
        hist_feat(turn_cos, bins=16, lo=-1.0, hi=1.0),
        hist_feat(turn_sin, bins=16, lo=-1.0, hi=1.0),
    ])

    return feat.astype(np.float32)


# ------------------------------------------------------------
# Build descriptors for focused subset
# ------------------------------------------------------------
desc_focus = np.vstack([ring_descriptor_paper_lite(r) for r in rings_focus]).astype(np.float32)

# standardize
mu = desc_focus.mean(axis=0, keepdims=True)
sd = desc_focus.std(axis=0, keepdims=True) + 1e-8
desc_focus_z = (desc_focus - mu) / sd

# L2 normalize for cosine similarity
norm = np.linalg.norm(desc_focus_z, axis=1, keepdims=True) + 1e-8
desc_focus_z = desc_focus_z / norm

print("Descriptor matrix:", desc_focus_z.shape)


# ------------------------------------------------------------
# Similarity + ranking
# ------------------------------------------------------------
sim_desc = desc_focus_z @ desc_focus_z.T
np.fill_diagonal(sim_desc, -1.0)

pred_rankings_desc = {}
for q in val_query_locals:
    pred_rankings_desc[q] = np.argsort(-sim_desc[q])[:10].tolist()

# baseline
focus_num_nodes = np.asarray([int(g.num_nodes_manual) for g in pyg_graphs_focus], dtype=np.int32)
baseline_rankings = {}
for q in val_query_locals:
    s = -np.abs(focus_num_nodes - focus_num_nodes[q]).astype(np.float32)
    s[q] = -1e9
    baseline_rankings[q] = np.argsort(-s)[:10].tolist()

def recall_at_k_ranked(pred_rankings, gt_local_map, query_local_ids, k=10):
    vals = []
    for q in query_local_ids:
        gt_set = set(gt_local_map[q][:k])
        pred_set = set(pred_rankings[q][:k])
        vals.append(len(gt_set & pred_set) / max(len(gt_set), 1))
    return float(np.mean(vals))

def hit_at_k_ranked(pred_rankings, gt_local_map, query_local_ids, k=10):
    vals = []
    for q in query_local_ids:
        gt_set = set(gt_local_map[q][:k])
        pred_set = set(pred_rankings[q][:k])
        vals.append(1.0 if len(gt_set & pred_set) > 0 else 0.0)
    return float(np.mean(vals))

r10_desc = recall_at_k_ranked(pred_rankings_desc, gt_local, val_query_locals, k=10)
h10_desc = hit_at_k_ranked(pred_rankings_desc, gt_local, val_query_locals, k=10)

r10_base = recall_at_k_ranked(baseline_rankings, gt_local, val_query_locals, k=10)
h10_base = hit_at_k_ranked(baseline_rankings, gt_local, val_query_locals, k=10)

print("\n=== PAPER-LITE DESCRIPTOR EVAL ===")
print(f"Node-count baseline Recall@10   : {r10_base:.4f}")
print(f"Node-count baseline Hit@10      : {h10_base:.4f}")
print(f"Paper-lite descriptor Recall@10 : {r10_desc:.4f}")
print(f"Paper-lite descriptor Hit@10    : {h10_desc:.4f}")

print("\n=== SAMPLE VAL QUERY EXAMPLES ===")
for q in val_query_locals[:3]:
    print(f"\nlocal_q={q} global_q={l2g[q]}")
    print("GT top10 globals   :", [l2g[x] for x in gt_local[q][:10]])
    print("Pred top10 globals :", [l2g[x] for x in pred_rankings_desc[q][:10]])

Descriptor matrix: (5775, 71)

=== PAPER-LITE DESCRIPTOR EVAL ===
Node-count baseline Recall@10   : 0.0000
Node-count baseline Hit@10      : 0.0000
Paper-lite descriptor Recall@10 : 0.0017
Paper-lite descriptor Hit@10    : 0.0172

=== SAMPLE VAL QUERY EXAMPLES ===

local_q=5738 global_q=187282
GT top10 globals   : [183778, 62218, 183799, 156861, 144868, 71191, 97469, 182612, 180700, 134169]
Pred top10 globals : [9874, 49611, 88055, 54870, 30998, 159814, 168900, 139177, 175213, 108155]

local_q=5624 global_q=187168
GT top10 globals   : [176959, 22051, 98011, 144662, 149799, 123303, 90397, 132996, 122587, 78485]
Pred top10 globals : [97535, 11084, 187311, 146039, 39868, 187160, 147329, 65317, 13514, 62296]

local_q=5567 global_q=187111
GT top10 globals   : [103601, 166163, 48071, 3843, 35951, 28125, 62149, 112588, 122421, 54898]
Pred top10 globals : [127952, 86042, 120194, 27329, 76483, 144060, 133904, 17375, 142450, 686]
